# ISAS26 ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â Seven experiments for hallway decoding

Notebook giÃƒÂ¡Ã‚Â»Ã‚Â¯ **A3** lÃƒÆ’Ã‚Â m stable-room backbone vÃƒÆ’Ã‚Â  triÃƒÂ¡Ã‚Â»Ã†â€™n khai bÃƒÂ¡Ã‚ÂºÃ‚Â£y hÃƒâ€ Ã‚Â°ÃƒÂ¡Ã‚Â»Ã¢â‚¬Âºng:

| ID | Experiment |
|---|---|
| H1 | Episode audit + inferred floor graph + graph jump repair |
| H2 | Multi-window stable anchors: 1s, 10s, 30s, 45s |
| H3 | Change-point anchor filling |
| H4 | Explicit-duration pruned Graph-HSMM |
| H5 | Room-conditional target-day RSSI calibration |
| H6 | Segment boundary classifier |
| H7 | Self-supervised causal GRU encoder |

Outer evaluation dÃƒÆ’Ã‚Â¹ng LODO bÃƒÂ¡Ã‚Â»Ã¢â‚¬Ëœn ngÃƒÆ’Ã‚Â y. H5 lÃƒÆ’Ã‚Â  transductive unsupervised adaptation vÃƒÆ’Ã‚Â¬ dÃƒÆ’Ã‚Â¹ng feature distribution vÃƒÆ’Ã‚Â  high-confidence prediction cÃƒÂ¡Ã‚Â»Ã‚Â§a test day, nhÃƒâ€ Ã‚Â°ng khÃƒÆ’Ã‚Â´ng dÃƒÆ’Ã‚Â¹ng test label. H2ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Å“H4 lÃƒÆ’Ã‚Â  offline decoding do cÃƒÆ’Ã‚Â³ thÃƒÂ¡Ã‚Â»Ã†â€™ dÃƒÆ’Ã‚Â¹ng stable anchor phÃƒÆ’Ã‚Â­a sau transition.


In [ ]:
# ============================================================
# 0. IMPORTS & CONFIG
# ============================================================

from __future__ import annotations

import json
import math
import os
import random
import re
import time
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.spatial.distance import jensenshannon

from sklearn.cluster import MiniBatchKMeans
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.feature_selection import mutual_info_classif
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
    precision_score,
    recall_score,
)
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_sample_weight

from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
plt.ioff()

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

# ---------- Data ----------
WINDOW_SECONDS = 3
FEATURE_DIR_CANDIDATES = [
Path.cwd() / "private_features",
    Path("/mnt/data"),
    Path("/content/drive/MyDrive/BLE_Localization_ISAS26/private_features"),
    Path.cwd(),
]
FEATURE_DIR = next(
    (
        p for p in FEATURE_DIR_CANDIDATES
        if (p / f"rf_window_features_{WINDOW_SECONDS}s.csv").exists()
    ),
    FEATURE_DIR_CANDIDATES[0],
)
FEATURE_FILE = FEATURE_DIR / f"rf_window_features_{WINDOW_SECONDS}s.csv"

OUTPUT_DIR = FEATURE_DIR / "ISAS26_1s_Eight_Ablations_Output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_COL = "room"
DATE_COL = "date"
TIME_COL = "window_ts"

# ---------- Feature policy ----------
USE_TIME_FEATURES = False
USE_POWER_FEATURES = False
USE_BEACON_ID_FEATURES = False
USE_B24_B25 = False

DROP_RSSI_STD_ARTIFACT = True
MAX_MISSING_RATIO = 0.98
NEAR_CONSTANT_DOMINANCE = 0.9995
CORR_THRESHOLD = 0.95
TOP_K = 100

MAX_RANK_SAMPLES = 8000
DAY_MI_PENALTY = 0.25
MISSING_PENALTY = 0.10
DROP_STRONG_DAY_DOMINANT = True
DAY_MI_MIN = 0.08
DAY_TO_ROOM_MI_RATIO = 1.25

USE_RELATIVE_RSSI = True
USE_SINGLE_PROTOTYPE = True

# ---------- Multi-prototype ----------
MAX_PROTOTYPES_PER_ROOM = 3
MIN_SAMPLES_PER_PROTOTYPE = 80
MIN_UNIQUE_PATTERNS_FOR_CLUSTERING = 3
KMEANS_BATCH_SIZE = 1024

# ---------- Targeted hard-class augmentation ----------
HARD_CLASS_AUGMENTATION = {
    "hallway": 1.00,
    "cafeteria": 0.15,
    "cleaning": 0.50,
    "nurse station": 0.05,
}
HARD_CLASS_MAX_NEW = {
    "hallway": 1200,
    "cafeteria": 900,
    "cleaning": 700,
    "nurse station": 600,
}
AUG_ALPHA_MIN = 0.70
AUG_ALPHA_MAX = 0.95
AUG_RSSI_JITTER_FRAC = 0.025
AUG_TRANSITION_JITTER_FRAC = 0.02

# ---------- XGBoost ----------
SEEDS = [42, 1042, 2042]
MODEL_N_JOBS = max(1, min(6, os.cpu_count() or 1))

BEST_XGB_PARAMS = {
    "n_estimators": 300,
    "max_depth": 2,
    "learning_rate": 0.019708414272057828,
    "subsample": 0.5534281837285242,
    "colsample_bytree": 0.5819630370366842,
    "min_child_weight": 7.991772382059395,
    "gamma": 1.3931793892618227,
    "reg_lambda": 28.679898648982864,
    "reg_alpha": 0.02726257973632696,
    "max_delta_step": 6.923947311989912,
}

# ---------- Sequential inference ----------
STANDARD_SMOOTH_WINDOW = 5
STANDARD_MARKOV_LAMBDA = 1.0

HALLWAY_SMOOTH_WINDOW = 3
HALLWAY_MARKOV_LAMBDA = 0.50
HALLWAY_GRAPH_MIX = 0.20
HALLWAY_EDGE_PRIOR = 20.0
HALLWAY_EMISSION_BOOST = 2.0

MARKOV_EPS = 1e-3
MAX_GAP_MULTIPLIER = 2.5

# ---------- Evaluation ----------
REPORT_PROTOCOLS = ["closed_set", "open_set"]
KEY_CLASSES = [
    "hallway",
    "cafeteria",
    "cleaning",
    "kitchen",
    "nurse station",
]
BOUNDARY_RADIUS = 2

# Strict decoding uses the true test-day class inventory and is retrospective.
# Keep False for deployment-faithful evaluation.
STRICT_CLOSED_SET_DECODING = False

# Set an integer such as 500 for quick debugging.
MAX_ROWS_PER_DAY: Optional[int] = None

# Set True only after the notebook has run successfully once.
SAVE_MODELS = False

print("FEATURE_FILE:", FEATURE_FILE)
print("OUTPUT_DIR :", OUTPUT_DIR)
print("N_JOBS     :", MODEL_N_JOBS)

# Injected by run_notebook_stage.py
FEATURE_DIR = Path.cwd() / "private_features"
FEATURE_FILE = FEATURE_DIR / f"rf_window_features_{WINDOW_SECONDS}s.csv"
OUTPUT_DIR = Path.cwd() / "artifacts" / "notebooks" / "experiments" / "3s"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
# ============================================================
# 0. HALLWAY EXPERIMENT CONFIG
# ============================================================

from dataclasses import dataclass
from collections import defaultdict
from typing import Any, Set

OUTPUT_DIR = FEATURE_DIR / "ISAS26_A3_Hallway_Seven_Experiments_Output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

HALLWAY_LABEL = "hallway"
RUN_EXPERIMENTS = {"H1", "H2", "H3", "H4", "H5", "H6"}
LONG_WINDOW_SEEDS = [42]
SAVE_EXPERIMENT_MODELS = False
RESUME_COMPLETED_FOLDS = False

MANUAL_ADJACENCY = {}

H1_REPAIR_RADIUS = 2
H1_MAX_REPAIR_POINTS = 3
H1_CONFIDENCE_MAX = 0.70

H2_LONG_WINDOWS = [10, 30, 45]
H2_MIN_SCALE_AGREEMENT = 2
H2_ANCHOR_CONFIDENCE_MIN = 0.58
H2_ANCHOR_MARGIN_MIN = 0.10
H2_MIN_ANCHOR_RUN = 3

H3_MAX_GAP_SECONDS = 180
H3_CHANGE_SCORE_MIN = 0.45
H3_UNSEEN_ROUTE_CHANGE_MIN = 0.65

H4_MAX_GAP_SECONDS = 240
H4_EDGE_SCORE_MARGIN = -0.25
H4_ROUTE_SMOOTHING = 1.0
H4_DURATION_SIGMA_FLOOR = 0.35

H5_ENABLE_TRANSDUCTIVE_ADAPTATION = True
H5_ANCHOR_CONFIDENCE_MIN = 0.75
H5_ANCHOR_MARGIN_MIN = 0.15
H5_MIN_ROOM_SUPPORT = 20
H5_SHRINKAGE_SUPPORT = 50
H5_MAX_ABS_RSSI_OFFSET = 10.0

H6_BOUNDARY_RADIUS = 2
H6_THRESHOLD_GRID = np.linspace(0.20, 0.80, 25)
H6_MIN_PRECISION = 0.15
H6_MAX_POSITIVE_WEIGHT = 8.0
H6_MIN_GAP_BOUNDARY_RATE = 0.25
H6_MIN_GAP_BOUNDARY_PROB = 0.45
H6_MAX_GAP_SECONDS = 180

H7_SSL_EPOCHS = 8
H7_SSL_SEQUENCE_LENGTH = 16
H7_SSL_STRIDE = 4
H7_SSL_MASK_RATE = 0.20
H7_SSL_HIDDEN_DIM = 32
H7_SSL_BATCH_SIZE = 128
H7_SSL_LEARNING_RATE = 1e-3
H7_SSL_MAX_SEQUENCES = 12000

KEY_CLASSES = ["hallway", "cafeteria", "cleaning", "kitchen", "nurse station"]

print("OUTPUT_DIR:", OUTPUT_DIR)

# Injected by run_notebook_stage.py
OUTPUT_DIR = Path.cwd() / "artifacts" / "notebooks" / "experiments" / "3s"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
# ============================================================
# 1. LOAD DATA
# ============================================================

META_COLS = {
    TARGET_COL,
    DATE_COL,
    TIME_COL,
    "timestamp",
    "window_start",
    "window_end",
    "start_time",
    "end_time",
    "day_group",
    "fold",
    "test_day",
    "y_true",
    "y_pred",
    "_source_file",
}

TIME_FEATURES = {
    "hour",
    "minute",
    "second",
    "dayofweek",
    "time_sin",
    "time_cos",
}

DIAGNOSTIC_PATTERNS = [
    "label_reliability",
    "label_confidence",
    "label_duration",
    "boundary_distance",
    "boundary_ratio",
    "is_boundary",
    "annotation",
    "overlap",
    "groundtruth",
    "ground_truth",
    "prev_room_gt",
]


def load_feature_data(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"KhÃƒÆ’Ã‚Â´ng tÃƒÆ’Ã‚Â¬m thÃƒÂ¡Ã‚ÂºÃ‚Â¥y feature file: {path}")

    data = pd.read_csv(path)

    required = {TARGET_COL, TIME_COL}
    missing = sorted(required - set(data.columns))
    if missing:
        raise ValueError(f"ThiÃƒÂ¡Ã‚ÂºÃ‚Â¿u required columns: {missing}")

    data[TIME_COL] = pd.to_datetime(data[TIME_COL], errors="coerce")

    if DATE_COL in data.columns:
        parsed_date = pd.to_datetime(data[DATE_COL], errors="coerce")
        data[DATE_COL] = parsed_date.dt.date.astype("string")
    else:
        data[DATE_COL] = data[TIME_COL].dt.date.astype("string")

    data[TARGET_COL] = data[TARGET_COL].astype("string").str.strip()
    data = data.dropna(subset=[TIME_COL, DATE_COL, TARGET_COL]).copy()
    data = data.sort_values([DATE_COL, TIME_COL]).reset_index(drop=True)

    if MAX_ROWS_PER_DAY is not None:
        data = (
            data.groupby(DATE_COL, group_keys=False)
                .head(int(MAX_ROWS_PER_DAY))
                .reset_index(drop=True)
        )

    return data


df = load_feature_data(FEATURE_FILE)

dataset_audit = (
    df.groupby(DATE_COL, observed=True)
      .agg(
          n_rows=(TARGET_COL, "size"),
          n_rooms=(TARGET_COL, "nunique"),
          start_time=(TIME_COL, "min"),
          end_time=(TIME_COL, "max"),
      )
      .reset_index()
)

dataset_audit.to_csv(OUTPUT_DIR / "dataset_audit.csv", index=False)
pd.crosstab(df[DATE_COL], df[TARGET_COL]).to_csv(
    OUTPUT_DIR / "class_distribution_by_day.csv"
)

print("Shape:", df.shape)
print("Days :", sorted(df[DATE_COL].astype(str).unique()))
print("Rooms:", sorted(df[TARGET_COL].astype(str).unique()))
display(dataset_audit)


In [ ]:
# ============================================================
# 2. BASE FEATURE POLICY AND RSSI-STD ARTIFACT CHECK
# ============================================================

def is_b24_b25_feature(name: str) -> bool:
    return bool(re.search(r"(?:^|_)B(?:24|25)(?:_|$)", str(name)))


def is_diagnostic_feature(name: str) -> bool:
    low = str(name).lower()
    return any(pattern in low for pattern in DIAGNOSTIC_PATTERNS)


def is_beacon_id_feature(name: str) -> bool:
    low = str(name).lower()
    return low.endswith("_id") or "_id_" in low


def is_per_beacon_rssi_std(name: str) -> bool:
    return bool(re.fullmatch(r"rssi_std_B\d{2}", str(name)))


def audit_rssi_std_artifact(data: pd.DataFrame) -> pd.DataFrame:
    std_cols = [c for c in data.columns if is_per_beacon_rssi_std(c)]
    rows = []

    if not std_cols:
        return pd.DataFrame()

    first_mask = data[std_cols[0]].isna()

    for c in std_cols:
        beacon = c.rsplit("_", 1)[-1]
        count_col = f"count_{beacon}"

        if count_col in data.columns:
            count_zero = (
                pd.to_numeric(data[count_col], errors="coerce")
                .fillna(0)
                .eq(0)
            )
            zero_nonmissing = int((count_zero & data[c].notna()).sum())
            zero_value_zero = int(
                (
                    count_zero
                    & pd.to_numeric(data[c], errors="coerce").eq(0)
                ).sum()
            )
        else:
            zero_nonmissing = np.nan
            zero_value_zero = np.nan

        rows.append({
            "feature": c,
            "missing_ratio": float(data[c].isna().mean()),
            "same_missing_mask_as_first": bool(data[c].isna().equals(first_mask)),
            "count_zero_nonmissing": zero_nonmissing,
            "count_zero_value_zero": zero_value_zero,
        })

    return pd.DataFrame(rows)


rssi_std_audit = audit_rssi_std_artifact(df)
rssi_std_audit.to_csv(
    OUTPUT_DIR / "rssi_std_feature_artifact_audit.csv",
    index=False,
)

if len(rssi_std_audit):
    RSSI_STD_ARTIFACT_CONFIRMED = (
        rssi_std_audit["same_missing_mask_as_first"].mean() >= 0.90
        and pd.to_numeric(
            rssi_std_audit["count_zero_nonmissing"],
            errors="coerce",
        ).fillna(0).sum() > 0
    )
else:
    RSSI_STD_ARTIFACT_CONFIRMED = False


def build_base_feature_columns(
    data: pd.DataFrame,
) -> Tuple[List[str], pd.DataFrame]:
    selected = []
    rows = []

    for c in data.columns:
        low = str(c).lower()

        if c in META_COLS:
            status = "metadata"
        elif not pd.api.types.is_numeric_dtype(data[c]):
            status = "non_numeric"
        elif not USE_TIME_FEATURES and c in TIME_FEATURES:
            status = "time_feature"
        elif not USE_POWER_FEATURES and low.startswith("power"):
            status = "power_feature"
        elif not USE_BEACON_ID_FEATURES and is_beacon_id_feature(c):
            status = "beacon_id"
        elif not USE_B24_B25 and is_b24_b25_feature(c):
            status = "inactive_B24_B25"
        elif is_diagnostic_feature(c):
            status = "diagnostic_or_label_leakage"
        elif (
            DROP_RSSI_STD_ARTIFACT
            and RSSI_STD_ARTIFACT_CONFIRMED
            and is_per_beacon_rssi_std(c)
        ):
            status = "rssi_std_extraction_artifact"
        else:
            status = "candidate"
            selected.append(c)

        rows.append({
            "feature": c,
            "status": status,
            "dtype": str(data[c].dtype),
            "missing_ratio": float(data[c].isna().mean()),
            "n_unique": int(data[c].nunique(dropna=True)),
        })

    return selected, pd.DataFrame(rows)


base_feature_cols, base_policy_audit = build_base_feature_columns(df)
base_policy_audit.to_csv(
    OUTPUT_DIR / "base_feature_policy_audit.csv",
    index=False,
)

print("RSSI std artifact confirmed:", RSSI_STD_ARTIFACT_CONFIRMED)
print("Base numeric candidates:", len(base_feature_cols))
display(base_policy_audit["status"].value_counts())


In [ ]:
# ============================================================
# 3. COMMON HELPERS
# ============================================================

def get_freq_cols(data: pd.DataFrame) -> List[str]:
    return [
        f"freq_B{i:02d}"
        for i in range(1, 24)
        if f"freq_B{i:02d}" in data.columns
    ]


def get_present_cols(data: pd.DataFrame) -> List[str]:
    return [
        f"present_B{i:02d}"
        for i in range(1, 24)
        if f"present_B{i:02d}" in data.columns
    ]


def normalize_rows(values: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    values = np.asarray(values, dtype=float)
    values = np.nan_to_num(values, nan=0.0, posinf=0.0, neginf=0.0)
    values = np.clip(values, 0.0, None)

    sums = values.sum(axis=1, keepdims=True)
    bad = sums[:, 0] <= eps

    if np.any(bad):
        values[bad] = 1.0 / values.shape[1]
        sums = values.sum(axis=1, keepdims=True)

    return values / np.maximum(sums, eps)


def normalize_probabilities(
    probabilities: np.ndarray,
    eps: float = 1e-12,
) -> np.ndarray:
    probabilities = np.asarray(probabilities, dtype=float)
    probabilities = np.nan_to_num(
        probabilities,
        nan=0.0,
        posinf=0.0,
        neginf=0.0,
    )
    probabilities = np.clip(probabilities, 0.0, None)

    row_sum = probabilities.sum(axis=1, keepdims=True)
    bad = row_sum[:, 0] <= eps

    if np.any(bad):
        probabilities[bad] = 1.0 / probabilities.shape[1]
        row_sum = probabilities.sum(axis=1, keepdims=True)

    return probabilities / np.maximum(row_sum, eps)


def split_contiguous_segments(
    timestamps: Sequence[pd.Timestamp],
    window_seconds: int = WINDOW_SECONDS,
) -> List[np.ndarray]:
    ts = pd.to_datetime(pd.Series(timestamps)).reset_index(drop=True)

    if len(ts) == 0:
        return []

    max_gap = max(
        float(window_seconds) * MAX_GAP_MULTIPLIER,
        float(window_seconds) + 1.0,
    )

    gaps = ts.diff().dt.total_seconds().fillna(0.0)
    segment_ids = (gaps > max_gap).cumsum()

    return [
        np.asarray(indices, dtype=int)
        for indices in segment_ids.groupby(segment_ids).groups.values()
    ]


def boundary_mask(
    data: pd.DataFrame,
    radius: int = BOUNDARY_RADIUS,
) -> np.ndarray:
    result = np.zeros(len(data), dtype=bool)

    for indices in split_contiguous_segments(data[TIME_COL]):
        labels = data.iloc[indices][TARGET_COL].astype(str).to_numpy()
        local = np.zeros(len(indices), dtype=bool)

        if len(labels) > 1:
            local[1:] |= labels[1:] != labels[:-1]
            local[:-1] |= labels[:-1] != labels[1:]

        expanded = local.copy()

        for offset in range(1, radius + 1):
            expanded[offset:] |= local[:-offset]
            expanded[:-offset] |= local[offset:]

        result[indices] = expanded

    return result


def sanitize_room_name(room: str) -> str:
    value = re.sub(r"[^A-Za-z0-9]+", "_", str(room).strip())
    return value.strip("_").lower()


In [ ]:
# ============================================================
# 4. STATIC FEATURES: RELATIVE RSSI + SINGLE PROTOTYPE
# ============================================================

def add_relative_rssi_features(data: pd.DataFrame) -> pd.DataFrame:
    out = data.copy()

    if not USE_RELATIVE_RSSI:
        return out

    mean_cols = [
        f"rssi_mean_B{i:02d}"
        for i in range(1, 24)
        if f"rssi_mean_B{i:02d}" in out.columns
    ]

    if not mean_cols:
        return out

    matrix = out[mean_cols].apply(
        pd.to_numeric,
        errors="coerce",
    ).to_numpy(dtype=float)

    has_signal = np.isfinite(matrix).any(axis=1)
    strongest = np.full(len(out), np.nan, dtype=float)

    if has_signal.any():
        strongest[has_signal] = np.nanmax(matrix[has_signal], axis=1)

    for idx, source_col in enumerate(mean_cols):
        beacon = source_col.rsplit("_", 1)[-1]
        out[f"rssi_rel_{beacon}"] = matrix[:, idx] - strongest

    return out


def fit_single_room_fingerprints(
    train_data: pd.DataFrame,
    freq_cols: Sequence[str],
) -> pd.DataFrame:
    if not freq_cols:
        return pd.DataFrame()

    grouped = (
        train_data.groupby(TARGET_COL, observed=True)[list(freq_cols)]
                  .mean()
    )

    return pd.DataFrame(
        normalize_rows(grouped.to_numpy(dtype=float)),
        index=grouped.index.astype(str),
        columns=list(freq_cols),
    )


def add_single_prototype_features(
    data: pd.DataFrame,
    fingerprints: pd.DataFrame,
    freq_cols: Sequence[str],
) -> pd.DataFrame:
    out = data.copy()

    if not USE_SINGLE_PROTOTYPE or fingerprints.empty:
        return out

    P = normalize_rows(out[list(freq_cols)].to_numpy(dtype=float))
    Q = normalize_rows(fingerprints[list(freq_cols)].to_numpy(dtype=float))

    p_norm = np.linalg.norm(P, axis=1, keepdims=True)
    q_norm = np.linalg.norm(Q, axis=1, keepdims=True).T
    cosine = (P @ Q.T) / np.maximum(p_norm * q_norm, 1e-12)

    js = np.empty((len(P), len(Q)), dtype=float)
    l1 = np.empty_like(js)
    log2 = np.log(2.0)

    for j in range(len(Q)):
        q = Q[j][None, :]
        midpoint = 0.5 * (P + q)

        kl_pm = np.sum(
            np.where(
                P > 0,
                P * (
                    np.log(np.maximum(P, 1e-12))
                    - np.log(np.maximum(midpoint, 1e-12))
                ) / log2,
                0.0,
            ),
            axis=1,
        )
        kl_qm = np.sum(
            np.where(
                q > 0,
                q * (
                    np.log(np.maximum(q, 1e-12))
                    - np.log(np.maximum(midpoint, 1e-12))
                ) / log2,
                0.0,
            ),
            axis=1,
        )

        js[:, j] = np.sqrt(np.maximum(0.5 * (kl_pm + kl_qm), 0.0))
        l1[:, j] = np.abs(P - q).sum(axis=1)

    cosine_sorted = np.sort(cosine, axis=1)
    js_sorted = np.sort(js, axis=1)
    l1_sorted = np.sort(l1, axis=1)

    out["fp_cos_best"] = cosine_sorted[:, -1]
    out["fp_cos_second"] = (
        cosine_sorted[:, -2] if cosine.shape[1] > 1 else cosine_sorted[:, -1]
    )
    out["fp_cos_margin"] = out["fp_cos_best"] - out["fp_cos_second"]

    out["fp_js_best"] = js_sorted[:, 0]
    out["fp_js_second"] = (
        js_sorted[:, 1] if js.shape[1] > 1 else js_sorted[:, 0]
    )
    out["fp_js_margin"] = out["fp_js_second"] - out["fp_js_best"]

    out["fp_l1_best"] = l1_sorted[:, 0]
    out["fp_l1_second"] = (
        l1_sorted[:, 1] if l1.shape[1] > 1 else l1_sorted[:, 0]
    )
    out["fp_l1_margin"] = out["fp_l1_second"] - out["fp_l1_best"]

    best_cos = cosine.argmax(axis=1)
    best_js = js.argmin(axis=1)
    out["fp_metric_agreement"] = (best_cos == best_js).astype(float)
    out["fp_match_confidence"] = out["fp_cos_margin"] * out["fp_js_margin"]

    return out


def engineer_static_fold(
    train_raw: pd.DataFrame,
    test_raw: pd.DataFrame,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    train = add_relative_rssi_features(train_raw)
    test = add_relative_rssi_features(test_raw)

    freq_cols = get_freq_cols(train)
    fingerprints = fit_single_room_fingerprints(train, freq_cols)

    train = add_single_prototype_features(train, fingerprints, freq_cols)
    test = add_single_prototype_features(test, fingerprints, freq_cols)

    return train, test, fingerprints


In [ ]:
# ============================================================
# 5. CAUSAL TRANSITION FEATURES
# ============================================================

TRANSITION_SCORE_COLS = [
    "transition_freq_js_prev",
    "transition_freq_l1_prev",
    "transition_presence_change_count",
    "transition_dominant_changed",
]


def add_causal_transition_features(data: pd.DataFrame) -> pd.DataFrame:
    out = data.copy().sort_values([DATE_COL, TIME_COL]).reset_index(drop=True)

    freq_cols = get_freq_cols(out)
    present_cols = get_present_cols(out)
    mean_cols = [
        f"rssi_mean_B{i:02d}"
        for i in range(1, 24)
        if f"rssi_mean_B{i:02d}" in out.columns
    ]

    for c in freq_cols:
        out[f"delta_{c}"] = 0.0

    for c in mean_cols:
        beacon = c.rsplit("_", 1)[-1]
        out[f"delta_rssi_mean_{beacon}"] = 0.0

    scalar_delta_cols = [
        c for c in [
            "active_beacon_count",
            "total_detections",
            "freq_entropy",
            "dominant_beacon_freq",
            "rssi_mean_all",
            "rssi_range_all",
        ]
        if c in out.columns
    ]

    for c in scalar_delta_cols:
        out[f"delta_{c}"] = 0.0

    out["transition_new_beacon_count"] = 0.0
    out["transition_disappeared_beacon_count"] = 0.0
    out["transition_presence_change_count"] = 0.0
    out["transition_dominant_changed"] = 0.0
    out["transition_freq_cosine_distance_prev"] = 0.0
    out["transition_freq_js_prev"] = 0.0
    out["transition_freq_l1_prev"] = 0.0

    for indices in split_contiguous_segments(out[TIME_COL]):
        if len(indices) <= 1:
            continue

        segment = out.iloc[indices]

        if freq_cols:
            freq = normalize_rows(
                segment[freq_cols].to_numpy(dtype=float)
            )
            prev = np.vstack([freq[0], freq[:-1]])

            for col_idx, c in enumerate(freq_cols):
                out.loc[indices, f"delta_{c}"] = freq[:, col_idx] - prev[:, col_idx]

            dot = np.sum(freq * prev, axis=1)
            norm = np.linalg.norm(freq, axis=1) * np.linalg.norm(prev, axis=1)
            cosine_distance = 1.0 - dot / np.maximum(norm, 1e-12)

            midpoint = 0.5 * (freq + prev)
            log2 = np.log(2.0)
            kl_a = np.sum(
                np.where(
                    freq > 0,
                    freq * (
                        np.log(np.maximum(freq, 1e-12))
                        - np.log(np.maximum(midpoint, 1e-12))
                    ) / log2,
                    0.0,
                ),
                axis=1,
            )
            kl_b = np.sum(
                np.where(
                    prev > 0,
                    prev * (
                        np.log(np.maximum(prev, 1e-12))
                        - np.log(np.maximum(midpoint, 1e-12))
                    ) / log2,
                    0.0,
                ),
                axis=1,
            )
            js = np.sqrt(np.maximum(0.5 * (kl_a + kl_b), 0.0))
            l1 = np.abs(freq - prev).sum(axis=1)

            dominant = freq.argmax(axis=1)
            dominant_prev = np.r_[dominant[0], dominant[:-1]]

            out.loc[indices, "transition_freq_cosine_distance_prev"] = cosine_distance
            out.loc[indices, "transition_freq_js_prev"] = js
            out.loc[indices, "transition_freq_l1_prev"] = l1
            out.loc[indices, "transition_dominant_changed"] = (
                dominant != dominant_prev
            ).astype(float)

        if present_cols:
            present = (
                segment[present_cols]
                .apply(pd.to_numeric, errors="coerce")
                .fillna(0.0)
                .to_numpy(dtype=float)
                > 0.5
            )
            prev_present = np.vstack([present[0], present[:-1]])

            out.loc[indices, "transition_new_beacon_count"] = (
                present & ~prev_present
            ).sum(axis=1)
            out.loc[indices, "transition_disappeared_beacon_count"] = (
                ~present & prev_present
            ).sum(axis=1)
            out.loc[indices, "transition_presence_change_count"] = (
                present != prev_present
            ).sum(axis=1)

        if mean_cols:
            rssi = (
                segment[mean_cols]
                .apply(pd.to_numeric, errors="coerce")
                .to_numpy(dtype=float)
            )
            prev_rssi = np.vstack([rssi[0], rssi[:-1]])
            delta = rssi - prev_rssi
            delta[~np.isfinite(delta)] = 0.0

            for col_idx, c in enumerate(mean_cols):
                beacon = c.rsplit("_", 1)[-1]
                out.loc[indices, f"delta_rssi_mean_{beacon}"] = delta[:, col_idx]

        for c in scalar_delta_cols:
            values = pd.to_numeric(segment[c], errors="coerce").to_numpy(dtype=float)
            prev_values = np.r_[values[0], values[:-1]]
            delta = values - prev_values
            delta[~np.isfinite(delta)] = 0.0
            out.loc[indices, f"delta_{c}"] = delta

        # Reset the first point of every segment.
        first = indices[0]
        transition_generated = [
            c for c in out.columns
            if c.startswith("delta_") or c.startswith("transition_")
        ]
        out.loc[first, transition_generated] = 0.0

    return out


In [ ]:
# ============================================================
# 7. FULL-SAFE CLEANER AND DRIFT-AWARE SELECTOR
# ============================================================

def sample_rank_rows(
    data: pd.DataFrame,
    max_rows: int,
    random_state: int,
) -> pd.DataFrame:
    if len(data) <= max_rows:
        return data.copy()

    days = sorted(data[DATE_COL].astype(str).unique())
    per_day = max(1, max_rows // max(len(days), 1))
    parts = []

    for offset, day in enumerate(days):
        part = data[data[DATE_COL].astype(str) == day]
        parts.append(
            part.sample(
                n=min(per_day, len(part)),
                random_state=random_state + offset,
            )
        )

    sampled = pd.concat(parts, axis=0)

    if len(sampled) > max_rows:
        sampled = sampled.sample(n=max_rows, random_state=random_state)

    return sampled.sort_index()


def remove_duplicate_columns(
    X: pd.DataFrame,
) -> Tuple[List[str], List[str]]:
    keep = []
    dropped = []
    seen: Dict[int, str] = {}

    for c in X.columns:
        hash_value = int(
            pd.util.hash_pandas_object(X[c], index=False).sum()
        )

        if hash_value in seen and X[c].equals(X[seen[hash_value]]):
            dropped.append(c)
        else:
            seen[hash_value] = c
            keep.append(c)

    return keep, dropped


def basic_train_only_cleaner(
    train_data: pd.DataFrame,
    candidate_cols: Sequence[str],
) -> Dict[str, object]:
    X = (
        train_data.reindex(columns=list(candidate_cols))
                  .replace([np.inf, -np.inf], np.nan)
    )

    working = []
    rows = []

    for c in X.columns:
        missing_ratio = float(X[c].isna().mean())
        n_unique = int(X[c].nunique(dropna=True))

        if X[c].isna().all():
            status = "all_nan"
        elif missing_ratio > MAX_MISSING_RATIO:
            status = "excessive_missing"
        elif n_unique <= 1:
            status = "constant"
        else:
            status = "working"
            working.append(c)

        rows.append({
            "feature": c,
            "status": status,
            "missing_ratio": missing_ratio,
            "n_unique": n_unique,
        })

    medians = X[working].median(numeric_only=True)
    X_fill = X[working].fillna(medians).fillna(0.0).astype(float)

    near_constant_keep = []
    near_constant_drop = []

    for c in X_fill.columns:
        counts = X_fill[c].value_counts(dropna=False)
        dominance = float(counts.iloc[0] / max(len(X_fill), 1))

        if dominance >= NEAR_CONSTANT_DOMINANCE:
            near_constant_drop.append(c)
        else:
            near_constant_keep.append(c)

    X_nc = X_fill[near_constant_keep]
    duplicate_keep, duplicate_drop = remove_duplicate_columns(X_nc)

    audit = pd.DataFrame(rows)
    audit["near_constant_dropped"] = audit["feature"].isin(near_constant_drop)
    audit["duplicate_dropped"] = audit["feature"].isin(duplicate_drop)

    return {
        "selected_features": duplicate_keep,
        "medians": medians,
        "audit": audit,
        "near_constant_dropped": near_constant_drop,
        "duplicate_dropped": duplicate_drop,
    }


def minmax01(values: pd.Series) -> pd.Series:
    values = pd.to_numeric(values, errors="coerce").fillna(0.0)
    low = float(values.min())
    high = float(values.max())

    if high - low <= 1e-12:
        return pd.Series(np.zeros(len(values)), index=values.index)

    return (values - low) / (high - low)


def is_discrete_feature(name: str) -> bool:
    low = str(name).lower()
    return (
        low.startswith("present_b")
        or low.startswith("top1_beacon")
        or low.startswith("top2_beacon")
        or low.startswith("top3_beacon")
        or is_beacon_id_feature(name)
    )


def fit_drift_aware_selector(
    train_data: pd.DataFrame,
    candidate_cols: Sequence[str],
    top_k: int = TOP_K,
    random_state: int = RANDOM_STATE,
) -> Dict[str, object]:
    cleaner = basic_train_only_cleaner(train_data, candidate_cols)
    clean_cols = cleaner["selected_features"]

    if not clean_cols:
        raise ValueError("No usable features after train-only cleaning.")

    rank_source = train_data[[TARGET_COL, DATE_COL] + clean_cols].copy()
    rank_source = sample_rank_rows(
        rank_source,
        MAX_RANK_SAMPLES,
        random_state,
    )

    medians = rank_source[clean_cols].median(numeric_only=True)
    X_rank = (
        rank_source[clean_cols]
        .replace([np.inf, -np.inf], np.nan)
        .fillna(medians)
        .fillna(0.0)
        .astype(float)
    )

    room_encoder = LabelEncoder()
    day_encoder = LabelEncoder()

    y_room = room_encoder.fit_transform(rank_source[TARGET_COL].astype(str))
    y_day = day_encoder.fit_transform(rank_source[DATE_COL].astype(str))

    discrete_mask = np.array(
        [is_discrete_feature(c) for c in clean_cols],
        dtype=bool,
    )

    mi_room = mutual_info_classif(
        X_rank,
        y_room,
        discrete_features=discrete_mask,
        random_state=random_state,
    )

    if len(np.unique(y_day)) > 1:
        mi_day = mutual_info_classif(
            X_rank,
            y_day,
            discrete_features=discrete_mask,
            random_state=random_state,
        )
    else:
        mi_day = np.zeros(len(clean_cols), dtype=float)

    extra_trees = ExtraTreesClassifier(
        n_estimators=250,
        max_features="sqrt",
        min_samples_leaf=2,
        class_weight="balanced",
        random_state=random_state,
        n_jobs=MODEL_N_JOBS,
    )
    extra_trees.fit(X_rank, y_room)

    ranking = pd.DataFrame({
        "feature": clean_cols,
        "mi_room": mi_room,
        "mi_day": mi_day,
        "extratrees_importance": extra_trees.feature_importances_,
        "missing_ratio": [
            float(train_data[c].isna().mean())
            for c in clean_cols
        ],
    })

    ranking["mi_room_norm"] = minmax01(ranking["mi_room"])
    ranking["mi_day_norm"] = minmax01(ranking["mi_day"])
    ranking["et_norm"] = minmax01(ranking["extratrees_importance"])
    ranking["day_to_room_mi_ratio"] = (
        ranking["mi_day"] / (ranking["mi_room"] + 1e-12)
    )

    ranking["strong_day_dominant"] = (
        (ranking["mi_day"] >= DAY_MI_MIN)
        & (
            ranking["mi_day"]
            > DAY_TO_ROOM_MI_RATIO * ranking["mi_room"]
        )
    )

    ranking["priority_score"] = (
        0.45 * ranking["mi_room_norm"]
        + 0.45 * ranking["et_norm"]
        - DAY_MI_PENALTY * ranking["mi_day_norm"]
        - MISSING_PENALTY * ranking["missing_ratio"]
    )

    eligible = (
        ranking[~ranking["strong_day_dominant"]].copy()
        if DROP_STRONG_DAY_DOMINANT
        else ranking.copy()
    )

    ordered = (
        eligible.sort_values(
            ["priority_score", "mi_room", "extratrees_importance"],
            ascending=[False, False, False],
        )["feature"]
        .tolist()
    )

    corr_source = (
        train_data[ordered]
        .replace([np.inf, -np.inf], np.nan)
        .fillna(train_data[ordered].median(numeric_only=True))
        .fillna(0.0)
    )

    if len(corr_source) > MAX_RANK_SAMPLES:
        corr_source = corr_source.sample(
            n=MAX_RANK_SAMPLES,
            random_state=random_state,
        )

    corr_matrix = corr_source.corr(method="spearman").abs()

    kept = []
    corr_drop_rows = []

    for feature in ordered:
        conflict = None
        conflict_value = np.nan

        for accepted in kept:
            value = corr_matrix.at[feature, accepted]

            if np.isfinite(value) and value >= CORR_THRESHOLD:
                conflict = accepted
                conflict_value = float(value)
                break

        if conflict is None:
            kept.append(feature)
        else:
            corr_drop_rows.append({
                "dropped_feature": feature,
                "kept_feature": conflict,
                "abs_spearman_corr": conflict_value,
            })

    selected = kept[: min(top_k, len(kept))]

    ranking["eligible_after_day_filter"] = ranking["feature"].isin(
        eligible["feature"]
    )
    ranking["kept_after_corr"] = ranking["feature"].isin(kept)
    ranking["selected_top_k"] = ranking["feature"].isin(selected)

    return {
        "selected_features": selected,
        "ranking": ranking.sort_values(
            ["priority_score", "mi_room"],
            ascending=[False, False],
        ),
        "cleaning_audit": cleaner["audit"],
        "correlation_dropped": pd.DataFrame(corr_drop_rows),
    }


def transform_features(
    data: pd.DataFrame,
    selected_features: Sequence[str],
) -> pd.DataFrame:
    return (
        data.reindex(columns=list(selected_features))
            .replace([np.inf, -np.inf], np.nan)
            .astype(float)
    )


In [ ]:
# ============================================================
# 9. XGBOOST PROBABILITY ENSEMBLE
# ============================================================

def make_xgb_model(
    n_classes: int,
    seed: int,
) -> XGBClassifier:
    return XGBClassifier(
        **BEST_XGB_PARAMS,
        objective="multi:softprob",
        num_class=n_classes,
        eval_metric="mlogloss",
        tree_method="hist",
        missing=np.nan,
        random_state=seed,
        n_jobs=MODEL_N_JOBS,
    )


def align_probabilities(
    probabilities: np.ndarray,
    source_classes: Sequence[str],
    target_classes: Sequence[str],
) -> np.ndarray:
    source_classes = list(map(str, source_classes))
    target_classes = list(map(str, target_classes))
    source_position = {c: idx for idx, c in enumerate(source_classes)}

    aligned = np.zeros(
        (len(probabilities), len(target_classes)),
        dtype=float,
    )

    for target_idx, class_name in enumerate(target_classes):
        if class_name in source_position:
            aligned[:, target_idx] = probabilities[
                :, source_position[class_name]
            ]

    return normalize_probabilities(aligned)


def fit_xgb_ensemble(
    X_train: pd.DataFrame,
    y_train: Sequence[str],
    X_test: pd.DataFrame,
    target_classes: Sequence[str],
    seeds: Sequence[int],
    save_dir: Optional[Path] = None,
) -> np.ndarray:
    label_encoder = LabelEncoder()
    y_encoded = label_encoder.fit_transform(
        pd.Series(y_train).astype(str)
    )
    source_classes = label_encoder.classes_.astype(str)

    sample_weight = compute_sample_weight(
        class_weight="balanced",
        y=pd.Series(y_train).astype(str),
    )

    probabilities = []

    for seed in seeds:
        model = make_xgb_model(
            n_classes=len(source_classes),
            seed=int(seed),
        )
        model.fit(
            X_train,
            y_encoded,
            sample_weight=sample_weight,
        )

        fold_probabilities = model.predict_proba(X_test)
        fold_probabilities = align_probabilities(
            fold_probabilities,
            source_classes,
            target_classes,
        )
        probabilities.append(fold_probabilities)

        if SAVE_MODELS and save_dir is not None:
            save_dir.mkdir(parents=True, exist_ok=True)
            joblib.dump(
                {
                    "model": model,
                    "classes": source_classes.tolist(),
                    "features": X_train.columns.tolist(),
                },
                save_dir / f"xgb_seed_{seed}.joblib",
            )

    return normalize_probabilities(
        np.mean(np.stack(probabilities, axis=0), axis=0)
    )


In [ ]:
# ============================================================
# 10. SMOOTHING AND DECODERS
# ============================================================

def smooth_segment_probabilities(
    probabilities: np.ndarray,
    window: int,
) -> np.ndarray:
    probabilities = normalize_probabilities(probabilities)

    if window <= 1 or len(probabilities) <= 1:
        return probabilities

    if window == 5:
        kernel = np.array([0.10, 0.15, 0.20, 0.25, 0.30], dtype=float)
    elif window == 3:
        kernel = np.array([0.20, 0.30, 0.50], dtype=float)
    else:
        kernel = np.arange(1, window + 1, dtype=float)
        kernel /= kernel.sum()

    output = np.zeros_like(probabilities, dtype=float)

    for idx in range(len(probabilities)):
        start = max(0, idx - window + 1)
        segment = probabilities[start : idx + 1]
        weights = kernel[-len(segment) :]
        weights = weights / weights.sum()
        output[idx] = (segment * weights[:, None]).sum(axis=0)

    return normalize_probabilities(output)


def smooth_by_segments(
    probabilities: np.ndarray,
    timestamps: Sequence[pd.Timestamp],
    window: int,
) -> np.ndarray:
    output = np.zeros_like(probabilities, dtype=float)

    for indices in split_contiguous_segments(timestamps):
        output[indices] = smooth_segment_probabilities(
            probabilities[indices],
            window=window,
        )

    return normalize_probabilities(output)


def transition_counts_from_train(
    train_data: pd.DataFrame,
    classes: Sequence[str],
) -> np.ndarray:
    classes = list(map(str, classes))
    position = {c: idx for idx, c in enumerate(classes)}
    counts = np.full(
        (len(classes), len(classes)),
        MARKOV_EPS,
        dtype=float,
    )

    for indices in split_contiguous_segments(train_data[TIME_COL]):
        labels = train_data.iloc[indices][TARGET_COL].astype(str).to_numpy()

        for idx in range(1, len(labels)):
            previous = labels[idx - 1]
            current = labels[idx]

            if previous in position and current in position:
                counts[position[previous], position[current]] += 1.0

    return counts


def fit_standard_transition(
    train_data: pd.DataFrame,
    classes: Sequence[str],
) -> np.ndarray:
    counts = transition_counts_from_train(train_data, classes)
    return counts / counts.sum(axis=1, keepdims=True)


def fit_hallway_graph_transition(
    train_data: pd.DataFrame,
    classes: Sequence[str],
) -> np.ndarray:
    classes = list(map(str, classes))
    counts = transition_counts_from_train(train_data, classes)
    empirical = counts / counts.sum(axis=1, keepdims=True)

    graph = np.zeros_like(empirical, dtype=float)
    np.fill_diagonal(graph, 1.0)

    if "hallway" in classes:
        hallway_idx = classes.index("hallway")

        raw_counts = transition_counts_from_train(train_data, classes)

        for idx, class_name in enumerate(classes):
            if idx == hallway_idx:
                continue

            connected = (
                raw_counts[idx, hallway_idx] > MARKOV_EPS
                or raw_counts[hallway_idx, idx] > MARKOV_EPS
            )

            if connected:
                graph[idx, hallway_idx] += HALLWAY_EDGE_PRIOR
                graph[hallway_idx, idx] += HALLWAY_EDGE_PRIOR

        graph[hallway_idx, hallway_idx] += HALLWAY_EDGE_PRIOR

    graph = graph / np.maximum(graph.sum(axis=1, keepdims=True), 1e-12)

    transition = (
        (1.0 - HALLWAY_GRAPH_MIX) * empirical
        + HALLWAY_GRAPH_MIX * graph
    )

    return transition / transition.sum(axis=1, keepdims=True)


def viterbi_decode(
    emissions: np.ndarray,
    transition: np.ndarray,
    markov_lambda: float,
) -> np.ndarray:
    emissions = normalize_probabilities(emissions)
    n_steps, n_classes = emissions.shape

    if n_steps == 0:
        return np.array([], dtype=int)

    log_emissions = np.log(np.clip(emissions, 1e-12, 1.0))
    log_transition = (
        markov_lambda
        * np.log(np.clip(transition, 1e-12, 1.0))
    )

    dynamic = np.full((n_steps, n_classes), -np.inf, dtype=float)
    backpointer = np.zeros((n_steps, n_classes), dtype=int)

    dynamic[0] = log_emissions[0] - math.log(n_classes)

    for step in range(1, n_steps):
        scores = dynamic[step - 1][:, None] + log_transition
        backpointer[step] = scores.argmax(axis=0)
        dynamic[step] = scores.max(axis=0) + log_emissions[step]

    path = np.zeros(n_steps, dtype=int)
    path[-1] = int(dynamic[-1].argmax())

    for step in range(n_steps - 2, -1, -1):
        path[step] = backpointer[step + 1, path[step + 1]]

    return path


def viterbi_by_segments(
    probabilities: np.ndarray,
    timestamps: Sequence[pd.Timestamp],
    transition: np.ndarray,
    markov_lambda: float,
) -> np.ndarray:
    path = np.zeros(len(probabilities), dtype=int)

    for indices in split_contiguous_segments(timestamps):
        path[indices] = viterbi_decode(
            probabilities[indices],
            transition,
            markov_lambda=markov_lambda,
        )

    return path


def fit_transition_score_scaler(
    train_data: pd.DataFrame,
) -> Dict[str, Tuple[float, float]]:
    scaler = {}

    for c in TRANSITION_SCORE_COLS:
        if c not in train_data.columns:
            continue

        values = pd.to_numeric(train_data[c], errors="coerce").dropna()

        if len(values) == 0:
            scaler[c] = (0.0, 1.0)
            continue

        low = float(values.quantile(0.50))
        high = float(values.quantile(0.95))

        if high - low <= 1e-12:
            high = low + 1.0

        scaler[c] = (low, high)

    return scaler


def hallway_transition_score(
    data: pd.DataFrame,
    scaler: Dict[str, Tuple[float, float]],
) -> np.ndarray:
    score_parts = []

    for c, (low, high) in scaler.items():
        values = (
            pd.to_numeric(data[c], errors="coerce")
            .fillna(low)
            .to_numpy(dtype=float)
        )
        normalized = np.clip((values - low) / (high - low), 0.0, 1.0)
        score_parts.append(normalized)

    if not score_parts:
        return np.zeros(len(data), dtype=float)

    return np.mean(np.column_stack(score_parts), axis=1)


def boost_hallway_emission(
    probabilities: np.ndarray,
    classes: Sequence[str],
    test_data: pd.DataFrame,
    scaler: Dict[str, Tuple[float, float]],
) -> np.ndarray:
    classes = list(map(str, classes))
    probabilities = normalize_probabilities(probabilities.copy())

    if "hallway" not in classes:
        return probabilities

    hallway_idx = classes.index("hallway")
    score = hallway_transition_score(test_data, scaler)

    probabilities[:, hallway_idx] *= (
        1.0 + HALLWAY_EMISSION_BOOST * score
    )

    return normalize_probabilities(probabilities)


def restrict_to_true_test_inventory(
    probabilities: np.ndarray,
    classes: Sequence[str],
    test_labels: Sequence[str],
) -> np.ndarray:
    if not STRICT_CLOSED_SET_DECODING:
        return probabilities

    allowed = set(map(str, pd.unique(pd.Series(test_labels).astype(str))))
    mask = np.array([str(c) in allowed for c in classes], dtype=float)

    restricted = probabilities * mask[None, :]
    return normalize_probabilities(restricted)


In [ ]:
# ============================================================
# 11. EVALUATION HELPERS
# ============================================================

ABLATION_DESCRIPTIONS = {
    "A0": "full_safe_static_argmax",
    "A1": "selected_static_argmax",
    "A2": "selected_static_prob_smooth",
    "A3": "selected_static_standard_viterbi",
    "A4": "selected_transition_argmax",
    "A5": "selected_transition_hallway_viterbi",
    "A6": "multi_prototype_hallway_viterbi",
    "A7": "multi_prototype_targeted_aug_hallway_viterbi",
}


def evaluate_protocols(
    y_true: Sequence[str],
    y_pred: Sequence[str],
    train_classes: Sequence[str],
    ablation_id: str,
    test_day: str,
) -> pd.DataFrame:
    y_true = pd.Series(y_true).astype(str).to_numpy()
    y_pred = pd.Series(y_pred).astype(str).to_numpy()
    train_class_set = set(map(str, train_classes))

    rows = []

    protocol_masks = {
        "open_set": np.ones(len(y_true), dtype=bool),
        "closed_set": np.array(
            [label in train_class_set for label in y_true],
            dtype=bool,
        ),
    }

    for protocol, mask in protocol_masks.items():
        if protocol not in REPORT_PROTOCOLS or not mask.any():
            continue

        true_subset = y_true[mask]
        pred_subset = y_pred[mask]
        labels = sorted(pd.unique(true_subset))

        rows.append({
            "ablation_id": ablation_id,
            "description": ABLATION_DESCRIPTIONS[ablation_id],
            "test_day": str(test_day),
            "protocol": protocol,
            "n_samples": int(mask.sum()),
            "n_labels": int(len(labels)),
            "accuracy": float(accuracy_score(true_subset, pred_subset)),
            "balanced_accuracy": float(
                balanced_accuracy_score(true_subset, pred_subset)
            ),
            "macro_f1": float(
                f1_score(
                    true_subset,
                    pred_subset,
                    labels=labels,
                    average="macro",
                    zero_division=0,
                )
            ),
            "weighted_f1": float(
                f1_score(
                    true_subset,
                    pred_subset,
                    labels=labels,
                    average="weighted",
                    zero_division=0,
                )
            ),
            "unseen_support": int(
                sum(label not in train_class_set for label in true_subset)
            ),
        })

    return pd.DataFrame(rows)


def class_metrics_table(
    y_true: Sequence[str],
    y_pred: Sequence[str],
    train_classes: Sequence[str],
    ablation_id: str,
    test_day: str,
) -> pd.DataFrame:
    y_true = pd.Series(y_true).astype(str).to_numpy()
    y_pred = pd.Series(y_pred).astype(str).to_numpy()
    train_class_set = set(map(str, train_classes))

    known_mask = np.array(
        [label in train_class_set for label in y_true],
        dtype=bool,
    )
    y_true = y_true[known_mask]
    y_pred = y_pred[known_mask]

    labels = sorted(pd.unique(y_true))

    precision, recall, f1, support = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=labels,
        zero_division=0,
    )

    table = pd.DataFrame({
        "class": labels,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "support": support,
    })
    table["ablation_id"] = ablation_id
    table["test_day"] = str(test_day)

    return table


def subset_metrics_table(
    test_data: pd.DataFrame,
    y_pred: Sequence[str],
    ablation_id: str,
    test_day: str,
) -> pd.DataFrame:
    y_true = test_data[TARGET_COL].astype(str).to_numpy()
    y_pred = pd.Series(y_pred).astype(str).to_numpy()
    boundary = boundary_mask(test_data)

    subsets = {
        "boundary": boundary,
        "stable": ~boundary,
        "hallway_true": y_true == "hallway",
        "cafeteria_true": y_true == "cafeteria",
        "nurse_station_true": y_true == "nurse station",
    }

    rows = []

    for subset, mask in subsets.items():
        if not mask.any():
            continue

        labels = sorted(pd.unique(y_true[mask]))

        rows.append({
            "ablation_id": ablation_id,
            "test_day": str(test_day),
            "subset": subset,
            "n_samples": int(mask.sum()),
            "accuracy": float(accuracy_score(y_true[mask], y_pred[mask])),
            "macro_f1": float(
                f1_score(
                    y_true[mask],
                    y_pred[mask],
                    labels=labels,
                    average="macro",
                    zero_division=0,
                )
            ),
        })

    return pd.DataFrame(rows)


def save_confusion_matrix(
    y_true: Sequence[str],
    y_pred: Sequence[str],
    output_path: Path,
    title: str,
) -> None:
    y_true = pd.Series(y_true).astype(str)
    y_pred = pd.Series(y_pred).astype(str)

    labels = sorted(set(y_true.unique()) | set(y_pred.unique()))
    matrix = confusion_matrix(y_true, y_pred, labels=labels)
    row_sum = matrix.sum(axis=1, keepdims=True)
    normalized = np.divide(
        matrix,
        row_sum,
        out=np.zeros_like(matrix, dtype=float),
        where=row_sum > 0,
    )

    pd.DataFrame(
        normalized,
        index=labels,
        columns=labels,
    ).to_csv(output_path.with_suffix(".csv"))

    size = max(9, min(18, 0.65 * len(labels)))
    plt.figure(figsize=(size, size))
    image = plt.imshow(normalized, aspect="auto", vmin=0, vmax=1)
    plt.colorbar(image, fraction=0.046, pad=0.04)
    plt.xticks(np.arange(len(labels)), labels, rotation=90, fontsize=8)
    plt.yticks(np.arange(len(labels)), labels, fontsize=8)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title(title)
    plt.tight_layout()
    plt.savefig(output_path.with_suffix(".png"), dpi=180, bbox_inches="tight")
    plt.close()


In [ ]:
# ============================================================
# 1. LOAD MULTI-WINDOW FEATURE FILES
# ============================================================

def load_window_feature_file(seconds):
    candidates = [
        FEATURE_DIR / f"rf_window_features_{seconds}s.csv",
        Path("/mnt/data") / f"rf_window_features_{seconds}s.csv",
        Path.cwd() / f"rf_window_features_{seconds}s.csv",
    ]
    path = next((p for p in candidates if p.exists()), None)
    if path is None:
        raise FileNotFoundError(f"Missing rf_window_features_{seconds}s.csv")

    data = pd.read_csv(path)
    data[TIME_COL] = pd.to_datetime(data[TIME_COL], errors="coerce")
    data[DATE_COL] = pd.to_datetime(data[DATE_COL], errors="coerce").dt.date.astype("string")
    data[TARGET_COL] = data[TARGET_COL].astype("string").str.strip()
    data = data.dropna(subset=[TIME_COL, DATE_COL, TARGET_COL])
    return data.sort_values([DATE_COL, TIME_COL]).reset_index(drop=True)

MULTI_WINDOW_DATA = {WINDOW_SECONDS: df}
for seconds in H2_LONG_WINDOWS:
    MULTI_WINDOW_DATA[seconds] = load_window_feature_file(seconds)

multi_window_audit = pd.DataFrame([
    {
        "window_seconds": seconds,
        "n_rows": len(data),
        "n_days": data[DATE_COL].nunique(),
        "n_rooms": data[TARGET_COL].nunique(),
    }
    for seconds, data in sorted(MULTI_WINDOW_DATA.items())
])
multi_window_audit.to_csv(OUTPUT_DIR / "multi_window_audit.csv", index=False)
display(multi_window_audit)


In [ ]:
# ============================================================
# 2. GENERIC A3 BUNDLE FOR 1s AND LONG WINDOWS
# ============================================================

@dataclass
class A3Bundle:
    window_seconds: int
    fingerprints: pd.DataFrame
    selected_features: list
    model_classes: list
    models: list
    transition: np.ndarray


def split_segments_generic(timestamps, window_seconds):
    ts = pd.to_datetime(pd.Series(timestamps)).reset_index(drop=True)
    if len(ts) == 0:
        return []
    max_gap = max(window_seconds * MAX_GAP_MULTIPLIER, window_seconds + 1.0)
    gaps = ts.diff().dt.total_seconds().fillna(0.0)
    segment_ids = (gaps > max_gap).cumsum()
    return [
        np.asarray(indices, dtype=int)
        for indices in segment_ids.groupby(segment_ids).groups.values()
    ]


def smooth_probabilities_generic(probabilities, timestamps, window_seconds):
    smooth_window = STANDARD_SMOOTH_WINDOW if window_seconds <= 5 else 3
    output = np.zeros_like(probabilities, dtype=float)
    for indices in split_segments_generic(timestamps, window_seconds):
        output[indices] = smooth_segment_probabilities(
            probabilities[indices],
            window=smooth_window,
        )
    return normalize_probabilities(output)


def fit_transition_generic(train_data, classes, window_seconds):
    classes = list(map(str, classes))
    position = {c: i for i, c in enumerate(classes)}
    counts = np.full((len(classes), len(classes)), MARKOV_EPS, dtype=float)

    for _, day_data in train_data.sort_values([DATE_COL, TIME_COL]).groupby(DATE_COL, observed=True):
        day_data = day_data.reset_index(drop=True)
        for indices in split_segments_generic(day_data[TIME_COL], window_seconds):
            labels = day_data.iloc[indices][TARGET_COL].astype(str).to_numpy()
            for i in range(1, len(labels)):
                a, b = labels[i - 1], labels[i]
                if a in position and b in position:
                    counts[position[a], position[b]] += 1.0

    return counts / counts.sum(axis=1, keepdims=True)


def viterbi_generic(probabilities, timestamps, transition, window_seconds):
    path = np.zeros(len(probabilities), dtype=int)
    for indices in split_segments_generic(timestamps, window_seconds):
        path[indices] = viterbi_decode(
            probabilities[indices],
            transition,
            markov_lambda=STANDARD_MARKOV_LAMBDA,
        )
    return path


def engineer_with_fingerprints(data, fingerprints):
    engineered = add_relative_rssi_features(data)
    return add_single_prototype_features(
        engineered,
        fingerprints,
        get_freq_cols(engineered),
    )


def fit_a3_bundle_generic(train_raw, window_seconds, seeds, random_state):
    train_static = add_relative_rssi_features(train_raw)
    freq_cols = get_freq_cols(train_static)
    fingerprints = fit_single_room_fingerprints(train_static, freq_cols)
    train_static = add_single_prototype_features(
        train_static,
        fingerprints,
        freq_cols,
    )

    candidates, _ = build_base_feature_columns(train_static)
    selector = fit_drift_aware_selector(
        train_static,
        candidates,
        top_k=TOP_K,
        random_state=random_state,
    )
    selected_features = selector["selected_features"]

    encoder = LabelEncoder()
    y_encoded = encoder.fit_transform(train_static[TARGET_COL].astype(str))
    classes = encoder.classes_.astype(str).tolist()
    weights = compute_sample_weight(
        class_weight="balanced",
        y=train_static[TARGET_COL].astype(str),
    )

    X_train = transform_features(train_static, selected_features)
    models = []
    for seed in seeds:
        model = make_xgb_model(len(classes), int(seed))
        model.fit(X_train, y_encoded, sample_weight=weights)
        models.append(model)

    transition = fit_transition_generic(train_raw, classes, window_seconds)

    return (
        A3Bundle(
            window_seconds=window_seconds,
            fingerprints=fingerprints,
            selected_features=list(selected_features),
            model_classes=classes,
            models=models,
            transition=transition,
        ),
        train_static,
    )


def predict_a3_bundle_generic(bundle, eval_raw):
    eval_static = engineer_with_fingerprints(eval_raw, bundle.fingerprints)
    X_eval = transform_features(eval_static, bundle.selected_features)
    probability_parts = []

    for model in bundle.models:
        probability_parts.append(
            align_probabilities(
                model.predict_proba(X_eval),
                bundle.model_classes,
                bundle.model_classes,
            )
        )

    probabilities_raw = normalize_probabilities(
        np.mean(np.stack(probability_parts, axis=0), axis=0)
    )
    probabilities_smooth = smooth_probabilities_generic(
        probabilities_raw,
        eval_static[TIME_COL],
        bundle.window_seconds,
    )
    path = viterbi_generic(
        probabilities_smooth,
        eval_static[TIME_COL],
        bundle.transition,
        bundle.window_seconds,
    )

    classes = np.asarray(bundle.model_classes, dtype=object)
    sorted_prob = np.sort(probabilities_smooth, axis=1)
    top1 = sorted_prob[:, -1]
    top2 = sorted_prob[:, -2] if probabilities_smooth.shape[1] > 1 else np.zeros(len(eval_static))
    entropy = -np.sum(
        probabilities_smooth * np.log(np.clip(probabilities_smooth, 1e-12, 1.0)),
        axis=1,
    ) / max(np.log(probabilities_smooth.shape[1]), 1e-12)

    return {
        "static": eval_static,
        "probabilities_raw": probabilities_raw,
        "probabilities_smooth": probabilities_smooth,
        "predictions": classes[path].astype(str),
        "top1_confidence": top1,
        "probability_margin": top1 - top2,
        "entropy": entropy,
        "classes": bundle.model_classes,
    }


In [ ]:
# ============================================================
# 3. H1 EPISODE AUDIT + TRAIN-ONLY FLOOR GRAPH
# ============================================================

@dataclass
class HallwayGraph:
    adjacency: dict
    route_counts: dict
    direct_counts: dict
    hallway_durations: list


def extract_label_episodes(data, label_col=TARGET_COL, window_seconds=1):
    rows = []
    for day, day_df in data.sort_values([DATE_COL, TIME_COL]).groupby(DATE_COL, observed=True):
        day_df = day_df.reset_index(drop=True)
        ts = pd.to_datetime(day_df[TIME_COL])
        labels = day_df[label_col].astype(str).to_numpy()
        if len(day_df) == 0:
            continue
        start = 0
        for i in range(1, len(day_df) + 1):
            boundary = i == len(day_df)
            if not boundary:
                gap = (ts.iloc[i] - ts.iloc[i - 1]).total_seconds()
                boundary = labels[i] != labels[i - 1] or gap > max(
                    window_seconds * MAX_GAP_MULTIPLIER,
                    window_seconds + 1,
                )
            if boundary:
                end = i - 1
                rows.append({
                    DATE_COL: str(day),
                    "label": labels[start],
                    "start_time": ts.iloc[start],
                    "end_time": ts.iloc[end],
                    "n_windows": end - start + 1,
                    "duration_seconds": (ts.iloc[end] - ts.iloc[start]).total_seconds() + window_seconds,
                })
                start = i

    episodes = pd.DataFrame(rows)
    if episodes.empty:
        return episodes

    episodes["previous_label"] = None
    episodes["next_label"] = None
    for _, indices in episodes.groupby(DATE_COL).groups.items():
        indices = list(indices)
        for pos, idx in enumerate(indices):
            if pos > 0:
                episodes.loc[idx, "previous_label"] = episodes.loc[indices[pos - 1], "label"]
            if pos + 1 < len(indices):
                episodes.loc[idx, "next_label"] = episodes.loc[indices[pos + 1], "label"]
    return episodes


def build_hallway_graph(train_raw):
    episodes = extract_label_episodes(train_raw)
    adjacency = defaultdict(set)
    route_counts = defaultdict(int)
    direct_counts = defaultdict(int)
    hallway_durations = []

    for _, episode in episodes.iterrows():
        label = str(episode["label"])
        if label == HALLWAY_LABEL:
            hallway_durations.append(float(episode["duration_seconds"]))
            a = episode["previous_label"]
            b = episode["next_label"]
            if pd.notna(a) and pd.notna(b):
                a, b = str(a), str(b)
                if a != HALLWAY_LABEL and b != HALLWAY_LABEL and a != b:
                    route_counts[(a, b)] += 1
                    route_counts[(b, a)] += 1
                    adjacency[a].add(b)
                    adjacency[b].add(a)

    for _, day_eps in episodes.groupby(DATE_COL, observed=True):
        day_eps = day_eps.reset_index(drop=True)
        for i in range(1, len(day_eps)):
            a = str(day_eps.loc[i - 1, "label"])
            b = str(day_eps.loc[i, "label"])
            if a != b and a != HALLWAY_LABEL and b != HALLWAY_LABEL:
                direct_counts[(a, b)] += 1
                direct_counts[(b, a)] += 1
                adjacency[a].add(b)
                adjacency[b].add(a)

    for room, neighbors in MANUAL_ADJACENCY.items():
        for neighbor in neighbors:
            if str(neighbor) == HALLWAY_LABEL:
                continue
            adjacency[str(room)].add(str(neighbor))
            adjacency[str(neighbor)].add(str(room))

    return HallwayGraph(
        adjacency={k: set(v) for k, v in adjacency.items()},
        route_counts=dict(route_counts),
        direct_counts=dict(direct_counts),
        hallway_durations=hallway_durations,
    ), episodes


def graph_has_edge(graph, a, b):
    return str(b) in graph.adjacency.get(str(a), set()) and str(a) != str(b)


def h1_graph_jump_repair(a3_pred, confidence, timestamps, graph):
    output = np.asarray(a3_pred, dtype=object).copy()
    confidence = np.asarray(confidence, dtype=float)
    for indices in split_segments_generic(timestamps, 1):
        for local_i in range(1, len(indices)):
            left_idx = indices[local_i - 1]
            right_idx = indices[local_i]
            a = str(output[left_idx])
            b = str(output[right_idx])
            if a == b or a == HALLWAY_LABEL or b == HALLWAY_LABEL:
                continue
            if not graph_has_edge(graph, a, b):
                continue
            lo = max(0, local_i - H1_REPAIR_RADIUS)
            hi = min(len(indices), local_i + H1_REPAIR_RADIUS + 1)
            candidates = indices[lo:hi]
            candidates = candidates[confidence[candidates] <= H1_CONFIDENCE_MAX]
            if len(candidates) == 0:
                continue
            selected = candidates[np.argsort(confidence[candidates])[:H1_MAX_REPAIR_POINTS]]
            output[selected] = HALLWAY_LABEL
    return output


In [ ]:
# ============================================================
# 4. H2 MULTI-WINDOW STABLE ANCHORS
# ============================================================

def align_long_result(one_second_test, long_test, long_result, seconds):
    left = one_second_test[[TIME_COL, DATE_COL]].copy().sort_values([DATE_COL, TIME_COL])
    right = long_test[[TIME_COL, DATE_COL]].copy()
    right[f"pred_{seconds}s"] = long_result["predictions"]
    right[f"conf_{seconds}s"] = long_result["top1_confidence"]
    right[f"margin_{seconds}s"] = long_result["probability_margin"]
    right = right.sort_values([DATE_COL, TIME_COL])

    parts = []
    for day, left_day in left.groupby(DATE_COL, observed=True):
        right_day = right[right[DATE_COL] == day]
        if right_day.empty:
            part = left_day.copy()
            part[f"pred_{seconds}s"] = None
            part[f"conf_{seconds}s"] = np.nan
            part[f"margin_{seconds}s"] = np.nan
        else:
            part = pd.merge_asof(
                left_day.sort_values(TIME_COL),
                right_day.drop(columns=[DATE_COL]).sort_values(TIME_COL),
                on=TIME_COL,
                direction="nearest",
                tolerance=pd.Timedelta(seconds=max(seconds, 2)),
            )
        parts.append(part)
    return pd.concat(parts).sort_index().reset_index(drop=True)


def apply_minimum_anchor_run(labels, valid, timestamps, minimum_run):
    labels = np.asarray(labels, dtype=object)
    valid = np.asarray(valid, dtype=bool)
    output = np.zeros_like(valid, dtype=bool)
    for indices in split_segments_generic(timestamps, 1):
        pos = 0
        while pos < len(indices):
            idx = indices[pos]
            if not valid[idx]:
                pos += 1
                continue
            label = labels[idx]
            end = pos + 1
            while end < len(indices):
                j = indices[end]
                if not valid[j] or labels[j] != label:
                    break
                end += 1
            if end - pos >= minimum_run:
                output[indices[pos:end]] = True
            pos = end
    return output


def build_multi_window_anchors(one_second_test, aligned_results):
    n = len(one_second_test)
    labels = np.full(n, None, dtype=object)
    conf = np.zeros(n)
    margin = np.zeros(n)
    agreement = np.zeros(n, dtype=int)

    for i in range(n):
        votes = defaultdict(list)
        margins = defaultdict(list)
        for seconds, aligned in aligned_results.items():
            label = aligned.loc[i, f"pred_{seconds}s"]
            value = aligned.loc[i, f"conf_{seconds}s"]
            value_margin = aligned.loc[i, f"margin_{seconds}s"]
            if pd.isna(label) or pd.isna(value):
                continue
            votes[str(label)].append(float(value))
            margins[str(label)].append(0.0 if pd.isna(value_margin) else float(value_margin))
        if not votes:
            continue
        best = max(votes, key=lambda key: (len(votes[key]), np.mean(votes[key])))
        labels[i] = best
        agreement[i] = len(votes[best])
        conf[i] = np.mean(votes[best])
        margin[i] = np.mean(margins[best])

    valid = (
        (agreement >= H2_MIN_SCALE_AGREEMENT)
        & (conf >= H2_ANCHOR_CONFIDENCE_MIN)
        & (margin >= H2_ANCHOR_MARGIN_MIN)
        & pd.Series(labels).notna().to_numpy()
    )
    valid = apply_minimum_anchor_run(
        labels,
        valid,
        one_second_test[TIME_COL],
        H2_MIN_ANCHOR_RUN,
    )
    return pd.DataFrame({
        "anchor_label": labels,
        "anchor_valid": valid,
        "anchor_confidence": conf,
        "anchor_margin": margin,
        "scale_agreement": agreement,
    })


def h2_anchor_stabilization(a3_pred, anchor_table):
    output = np.asarray(a3_pred, dtype=object).copy()
    mask = anchor_table["anchor_valid"].to_numpy(dtype=bool)
    output[mask] = anchor_table.loc[mask, "anchor_label"].astype(str).to_numpy()
    return output


def extract_anchor_runs(anchor_table, timestamps):
    labels = anchor_table["anchor_label"].to_numpy(dtype=object)
    valid = anchor_table["anchor_valid"].to_numpy(dtype=bool)
    runs = []
    for indices in split_segments_generic(timestamps, 1):
        pos = 0
        while pos < len(indices):
            idx = indices[pos]
            if not valid[idx]:
                pos += 1
                continue
            label = str(labels[idx])
            end = pos + 1
            while end < len(indices):
                j = indices[end]
                if not valid[j] or str(labels[j]) != label:
                    break
                end += 1
            runs.append({"label": label, "start": int(indices[pos]), "end": int(indices[end - 1])})
            pos = end
    return runs


def candidate_anchor_gaps(anchor_table, timestamps):
    runs = extract_anchor_runs(anchor_table, timestamps)
    ts = pd.to_datetime(pd.Series(timestamps)).reset_index(drop=True)
    gaps = []
    for i in range(1, len(runs)):
        a, b = runs[i - 1], runs[i]
        if a["label"] == b["label"]:
            continue
        start = a["end"] + 1
        end = b["start"] - 1
        if end < start:
            continue
        duration = (ts.iloc[end] - ts.iloc[start]).total_seconds() + 1.0
        gaps.append({
            "from_room": a["label"],
            "to_room": b["label"],
            "start": start,
            "end": end,
            "duration_seconds": duration,
        })
    return gaps


In [ ]:
# ============================================================
# 5. H3 CHANGE-POINT FILLING + H4 GRAPH-HSMM
# ============================================================

CHANGE_FEATURES = [
    "transition_freq_js_prev",
    "transition_freq_l1_prev",
    "transition_freq_cosine_distance_prev",
    "transition_presence_change_count",
    "transition_new_beacon_count",
    "transition_disappeared_beacon_count",
    "transition_dominant_changed",
]


def fit_change_scaler(train_transition):
    scaler = {}
    for feature in CHANGE_FEATURES:
        if feature not in train_transition.columns:
            continue
        values = pd.to_numeric(train_transition[feature], errors="coerce").dropna()
        if len(values) == 0:
            scaler[feature] = (0.0, 1.0)
        else:
            low = float(values.quantile(0.50))
            high = float(values.quantile(0.95))
            scaler[feature] = (low, high if high - low > 1e-12 else low + 1.0)
    return scaler


def compute_change_score(test_transition, scaler, entropy, margin):
    parts = []
    for feature, (low, high) in scaler.items():
        values = pd.to_numeric(test_transition[feature], errors="coerce").fillna(low).to_numpy(float)
        parts.append(np.clip((values - low) / (high - low), 0.0, 1.0))
    parts.append(np.clip(np.asarray(entropy, float), 0.0, 1.0))
    parts.append(np.clip(1.0 - np.asarray(margin, float), 0.0, 1.0))
    return np.mean(np.column_stack(parts), axis=1)


def h3_change_point_fill(base_pred, anchor_table, timestamps, change_score, graph):
    output = np.asarray(base_pred, dtype=object).copy()
    rows = []
    for gap in candidate_anchor_gaps(anchor_table, timestamps):
        indices = np.arange(gap["start"], gap["end"] + 1)
        mean_change = float(np.mean(change_score[indices]))
        route_seen = graph_has_edge(graph, gap["from_room"], gap["to_room"])
        accept = (
            gap["duration_seconds"] <= H3_MAX_GAP_SECONDS
            and (
                (route_seen and mean_change >= H3_CHANGE_SCORE_MIN)
                or ((not route_seen) and mean_change >= H3_UNSEEN_ROUTE_CHANGE_MIN)
            )
        )
        if accept:
            output[indices] = HALLWAY_LABEL
        rows.append({**gap, "route_seen": route_seen, "mean_change_score": mean_change, "accepted": accept})
    return output, pd.DataFrame(rows)


def fit_lognormal_duration(durations):
    values = np.asarray([max(float(x), 1.0) for x in durations], dtype=float)
    if len(values) == 0:
        return np.log(10.0), 1.0
    logs = np.log(values)
    return float(np.mean(logs)), max(float(np.std(logs)), H4_DURATION_SIGMA_FLOOR)


def lognormal_logpdf(duration, mu, sigma):
    duration = max(float(duration), 1e-6)
    return float(
        -np.log(duration * sigma * np.sqrt(2.0 * np.pi))
        - ((np.log(duration) - mu) ** 2) / (2.0 * sigma ** 2)
    )


def route_log_prior(graph, room_a, room_b):
    count = graph.route_counts.get((str(room_a), str(room_b)), 0)
    outgoing = sum(v for (source, _), v in graph.route_counts.items() if source == str(room_a))
    degree = max(len(graph.adjacency.get(str(room_a), set())), 1)
    probability = (count + H4_ROUTE_SMOOTHING) / (outgoing + H4_ROUTE_SMOOTHING * degree)
    return float(np.log(max(probability, 1e-12)))


def h4_graph_hsmm(base_pred, anchor_table, timestamps, change_score, a3_prob, classes, graph):
    output = np.asarray(base_pred, dtype=object).copy()
    class_pos = {str(c): i for i, c in enumerate(classes)}
    mu, sigma = fit_lognormal_duration(graph.hallway_durations)
    rows = []

    for gap in candidate_anchor_gaps(anchor_table, timestamps):
        indices = np.arange(gap["start"], gap["end"] + 1)
        if len(indices) == 0 or gap["duration_seconds"] > H4_MAX_GAP_SECONDS:
            continue
        uncertainty = 1.0 - np.max(a3_prob[indices], axis=1)
        edge_emission = float(np.mean(np.log(np.clip(0.5 * change_score[indices] + 0.5 * uncertainty, 1e-6, 1.0))))
        room_cols = [class_pos[r] for r in [str(gap["from_room"]), str(gap["to_room"])] if r in class_pos]
        if room_cols:
            room_emission = float(np.mean(np.log(np.clip(np.max(a3_prob[np.ix_(indices, room_cols)], axis=1), 1e-6, 1.0))))
        else:
            room_emission = np.log(1e-6)
        duration_score = lognormal_logpdf(gap["duration_seconds"], mu, sigma)
        route_score = route_log_prior(graph, gap["from_room"], gap["to_room"])
        edge_score = edge_emission + (duration_score + route_score) / max(len(indices), 1)
        margin_score = edge_score - room_emission
        accept = graph_has_edge(graph, gap["from_room"], gap["to_room"]) and margin_score >= H4_EDGE_SCORE_MARGIN
        if accept:
            output[indices] = HALLWAY_LABEL
        rows.append({**gap, "edge_emission": edge_emission, "room_emission": room_emission, "duration_score": duration_score, "route_score": route_score, "edge_minus_room": margin_score, "accepted": accept})
    return output, pd.DataFrame(rows)


In [ ]:
# ============================================================
# 6. H5 ROOM-CONDITIONAL TARGET-DAY RSSI CALIBRATION
# ============================================================

def estimate_room_offsets(train_raw, test_raw, baseline_result):
    mean_cols = [f"rssi_mean_B{i:02d}" for i in range(1, 24) if f"rssi_mean_B{i:02d}" in train_raw.columns]
    source = train_raw.groupby(TARGET_COL, observed=True)[mean_cols].median()
    stable = (
        (baseline_result["top1_confidence"] >= H5_ANCHOR_CONFIDENCE_MIN)
        & (baseline_result["probability_margin"] >= H5_ANCHOR_MARGIN_MIN)
        & (np.asarray(baseline_result["predictions"], str) != HALLWAY_LABEL)
    )
    target = test_raw.loc[stable].copy()
    target["_predicted_room"] = np.asarray(baseline_result["predictions"], str)[stable]

    raw_rows = []
    by_beacon = defaultdict(list)
    for room, room_df in target.groupby("_predicted_room", observed=True):
        room = str(room)
        if room not in source.index:
            continue
        for column in mean_cols:
            values = pd.to_numeric(room_df[column], errors="coerce").dropna()
            if len(values) < H5_MIN_ROOM_SUPPORT or pd.isna(source.loc[room, column]):
                continue
            offset = float(np.clip(values.median() - source.loc[room, column], -H5_MAX_ABS_RSSI_OFFSET, H5_MAX_ABS_RSSI_OFFSET))
            by_beacon[column].append(offset)
            raw_rows.append({"room": room, "feature": column, "support": len(values), "raw_offset": offset})

    global_offset = {column: float(np.median(values)) for column, values in by_beacon.items() if values}
    offsets = {}
    for row in raw_rows:
        shrink = row["support"] / (row["support"] + H5_SHRINKAGE_SUPPORT)
        final = shrink * row["raw_offset"] + (1.0 - shrink) * global_offset.get(row["feature"], 0.0)
        offsets[(row["room"], row["feature"])] = float(final)
        row["global_offset"] = global_offset.get(row["feature"], 0.0)
        row["shrinkage_weight"] = shrink
        row["final_offset"] = final
    return offsets, pd.DataFrame(raw_rows)


def apply_room_offsets(test_raw, baseline_predictions, offsets):
    # Deep copy the DataFrame and explicitly allocate writable NumPy arrays.
    # Some pandas/NumPy combinations return a read-only view from to_numpy().
    calibrated = test_raw.copy(deep=True)
    baseline_predictions = np.asarray(
        baseline_predictions,
        dtype=str,
    )

    if len(baseline_predictions) != len(calibrated):
        raise ValueError(
            "baseline_predictions and test_raw must have the same length: "
            f"{len(baseline_predictions)} != {len(calibrated)}"
        )

    for beacon in range(1, 24):
        mean_col = f"rssi_mean_B{beacon:02d}"

        if mean_col not in calibrated.columns:
            continue

        row_offsets = np.asarray(
            [
                offsets.get((str(room), mean_col), 0.0)
                for room in baseline_predictions
            ],
            dtype=np.float64,
        )

        for metric in ["mean", "median", "min", "max"]:
            column = f"rssi_{metric}_B{beacon:02d}"

            if column not in calibrated.columns:
                continue

            values = (
                pd.to_numeric(
                    calibrated[column],
                    errors="coerce",
                )
                .to_numpy(
                    dtype=np.float64,
                    copy=True,
                )
            )

            valid = np.isfinite(values) & np.isfinite(row_offsets)

            # Avoid in-place mutation on a possible read-only pandas-backed view.
            corrected = values.copy()
            corrected[valid] = (
                corrected[valid]
                - row_offsets[valid]
            )

            calibrated[column] = corrected

    return calibrated


In [ ]:
# ============================================================
# 7. H6 SEGMENT BOUNDARY CLASSIFIER
# ============================================================

BOUNDARY_BASE_FEATURES = [
    "active_beacon_count",
    "total_detections",
    "freq_entropy",
    "dominant_beacon_freq",
    "rssi_mean_all",
    "rssi_range_all",
    "rssi_iqr_all",
]


def make_boundary_target(data):
    labels = data[TARGET_COL].astype(str).to_numpy()
    target = labels == HALLWAY_LABEL
    for indices in split_segments_generic(data[TIME_COL], 1):
        local = labels[indices]
        changes = np.zeros(len(indices), dtype=bool)
        if len(indices) > 1:
            changes[1:] |= local[1:] != local[:-1]
            changes[:-1] |= local[:-1] != local[1:]
        expanded = changes.copy()
        for offset in range(1, H6_BOUNDARY_RADIUS + 1):
            expanded[offset:] |= changes[:-offset]
            expanded[:-offset] |= changes[offset:]
        target[indices] |= expanded
    return target.astype(int)


def get_boundary_features(data):
    return sorted(set([
        column
        for column in data.columns
        if (
            column.startswith("transition_")
            or column.startswith("delta_")
            or column in BOUNDARY_BASE_FEATURES
        )
        and pd.api.types.is_numeric_dtype(data[column])
    ]))


def make_boundary_model(positive_weight, seed):
    return XGBClassifier(
        n_estimators=250,
        max_depth=3,
        learning_rate=0.03,
        subsample=0.75,
        colsample_bytree=0.75,
        min_child_weight=4,
        reg_lambda=10.0,
        reg_alpha=0.1,
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        scale_pos_weight=float(positive_weight),
        random_state=seed,
        n_jobs=MODEL_N_JOBS,
    )


def select_boundary_threshold_lodo(train_transition, features):
    if not features:
        raise ValueError(
            "H6: khÃƒÆ’Ã‚Â´ng cÃƒÆ’Ã‚Â³ boundary features sau feature construction."
        )

    parts = []
    skipped_inner_days = []

    for inner_day in sorted(
        train_transition[DATE_COL].astype(str).unique()
    ):
        inner_train = train_transition[
            train_transition[DATE_COL].astype(str) != inner_day
        ].copy()

        inner_val = train_transition[
            train_transition[DATE_COL].astype(str) == inner_day
        ].copy()

        if inner_train.empty or inner_val.empty:
            skipped_inner_days.append(
                {
                    "inner_day": inner_day,
                    "reason": "empty_train_or_validation",
                }
            )
            continue

        y_train = make_boundary_target(inner_train)
        y_val = make_boundary_target(inner_val)

        if len(np.unique(y_train)) < 2:
            skipped_inner_days.append(
                {
                    "inner_day": inner_day,
                    "reason": "one_class_in_inner_train",
                }
            )
            continue

        positive = max(int(y_train.sum()), 1)
        negative = max(int((1 - y_train).sum()), 1)

        weight = min(
            H6_MAX_POSITIVE_WEIGHT,
            math.sqrt(negative / positive),
        )

        model = make_boundary_model(
            positive_weight=weight,
            seed=RANDOM_STATE,
        )

        model.fit(
            transform_features(inner_train, features),
            y_train,
        )

        probability = model.predict_proba(
            transform_features(inner_val, features)
        )[:, 1]

        parts.append(
            pd.DataFrame(
                {
                    "inner_day": str(inner_day),
                    "y_true": y_val.astype(int),
                    "probability": np.asarray(
                        probability,
                        dtype=float,
                    ),
                }
            )
        )

    if not parts:
        raise ValueError(
            "H6: khÃƒÆ’Ã‚Â´ng tÃƒÂ¡Ã‚ÂºÃ‚Â¡o Ãƒâ€žÃ¢â‚¬ËœÃƒâ€ Ã‚Â°ÃƒÂ¡Ã‚Â»Ã‚Â£c inner-LODO OOF predictions. "
            f"Skipped folds: {skipped_inner_days}"
        )

    oof = pd.concat(parts, ignore_index=True)

    rows = []

    for threshold in H6_THRESHOLD_GRID:
        pred = (
            oof["probability"].to_numpy(dtype=float)
            >= float(threshold)
        ).astype(int)

        rows.append(
            {
                "threshold": float(threshold),
                "precision": float(
                    precision_score(
                        oof["y_true"],
                        pred,
                        zero_division=0,
                    )
                ),
                "recall": float(
                    recall_score(
                        oof["y_true"],
                        pred,
                        zero_division=0,
                    )
                ),
                "f1": float(
                    f1_score(
                        oof["y_true"],
                        pred,
                        zero_division=0,
                    )
                ),
                "positive_rate": float(pred.mean()),
                "n_oof": int(len(oof)),
                "true_positive_support": int(
                    oof["y_true"].sum()
                ),
                "n_inner_folds_used": int(
                    oof["inner_day"].nunique()
                ),
            }
        )

    results = pd.DataFrame(rows)

    valid = results[
        results["precision"] >= H6_MIN_PRECISION
    ]

    selection_pool = (
        valid
        if not valid.empty
        else results
    )

    best = (
        selection_pool.sort_values(
            ["f1", "precision", "threshold"],
            ascending=[False, False, False],
        )
        .iloc[0]
    )

    results["selected"] = (
        np.isclose(
            results["threshold"],
            float(best["threshold"]),
        )
    )

    return float(best["threshold"]), results

def fit_boundary_model(train_transition, features):
    y = make_boundary_target(train_transition)
    positive = max(int(y.sum()), 1)
    negative = max(int((1 - y).sum()), 1)
    weight = min(H6_MAX_POSITIVE_WEIGHT, math.sqrt(negative / positive))
    model = make_boundary_model(weight, RANDOM_STATE)
    model.fit(transform_features(train_transition, features), y)
    return model, weight


def smooth_binary_probability(probability, timestamps, window=3):
    output = np.zeros(len(probability), dtype=float)
    kernel = np.array([0.20, 0.30, 0.50]) if window == 3 else np.arange(1, window + 1, dtype=float)
    kernel = kernel / kernel.sum()
    for indices in split_segments_generic(timestamps, 1):
        local = np.asarray(probability)[indices]
        for i in range(len(local)):
            start = max(0, i - window + 1)
            weights = kernel[-(i - start + 1):]
            weights = weights / weights.sum()
            output[indices[i]] = np.sum(local[start:i + 1] * weights)
    return output


def h6_boundary_decode(base_pred, anchor_table, timestamps, boundary_probability, threshold, graph):
    output = np.asarray(base_pred, dtype=object).copy()
    binary = boundary_probability >= threshold
    rows = []
    for gap in candidate_anchor_gaps(anchor_table, timestamps):
        indices = np.arange(gap["start"], gap["end"] + 1)
        mean_prob = float(np.mean(boundary_probability[indices]))
        positive_rate = float(np.mean(binary[indices]))
        route_seen = graph_has_edge(graph, gap["from_room"], gap["to_room"])
        accept = (
            gap["duration_seconds"] <= H6_MAX_GAP_SECONDS
            and route_seen
            and positive_rate >= H6_MIN_GAP_BOUNDARY_RATE
            and mean_prob >= H6_MIN_GAP_BOUNDARY_PROB
        )
        if accept:
            output[indices] = HALLWAY_LABEL
        rows.append({**gap, "route_seen": route_seen, "mean_boundary_probability": mean_prob, "boundary_positive_rate": positive_rate, "accepted": accept})
    return output, pd.DataFrame(rows)


In [ ]:
# ============================================================
# 8. H7 SELF-SUPERVISED CAUSAL GRU ÃƒÂ¢Ã¢â€šÂ¬Ã¢â‚¬Â FIXED V2
# ============================================================

def run_ssl_temporal_experiment(
    bundle,
    train_static,
    test_static,
):
    """
    Self-supervised causal GRU using masked reconstruction.

    Important corrections:
      1. Uses only outer-train sequences for SSL fitting.
      2. Uses the same fixed causal context length during
         pretraining and embedding extraction.
      3. Does not feed an entire day into a GRU trained only on
         short 16-step sequences.
      4. Checks feature consistency, finite values, sequence count,
         prediction length and training loss.
    """
    try:
        import torch
        import torch.nn as nn
        from torch.utils.data import DataLoader, TensorDataset
    except Exception as exc:
        print("H7 skipped, PyTorch unavailable:", exc)
        return None

    if train_static.empty or test_static.empty:
        print("H7 skipped: empty train or test data.")
        return None

    torch.manual_seed(RANDOM_STATE)
    np.random.seed(RANDOM_STATE)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(RANDOM_STATE)

    # --------------------------------------------------------
    # Input features
    # --------------------------------------------------------
    features = (
        get_freq_cols(train_static)
        + get_present_cols(train_static)
    )

    features += [
        column
        for column in [
            "active_beacon_count",
            "total_detections",
            "freq_entropy",
            "dominant_beacon_freq",
        ]
        if column in train_static.columns
    ]

    # Preserve order and remove duplicates.
    features = list(dict.fromkeys(features))

    features = [
        column
        for column in features
        if column in test_static.columns
    ]

    if not features:
        print("H7 skipped: no shared SSL input features.")
        return None

    train_matrix = (
        train_static[features]
        .apply(pd.to_numeric, errors="coerce")
        .fillna(0.0)
        .to_numpy(
            dtype=np.float32,
            copy=True,
        )
    )

    test_matrix = (
        test_static[features]
        .apply(pd.to_numeric, errors="coerce")
        .fillna(0.0)
        .to_numpy(
            dtype=np.float32,
            copy=True,
        )
    )

    if (
        train_matrix.ndim != 2
        or test_matrix.ndim != 2
        or train_matrix.shape[1] != test_matrix.shape[1]
    ):
        raise ValueError(
            "H7: train/test SSL matrices have incompatible shapes: "
            f"{train_matrix.shape} vs {test_matrix.shape}"
        )

    train_matrix = np.nan_to_num(
        train_matrix,
        nan=0.0,
        posinf=0.0,
        neginf=0.0,
    )
    test_matrix = np.nan_to_num(
        test_matrix,
        nan=0.0,
        posinf=0.0,
        neginf=0.0,
    )

    mean = train_matrix.mean(
        axis=0,
        keepdims=True,
    )
    std = train_matrix.std(
        axis=0,
        keepdims=True,
    )
    std[std < 1e-6] = 1.0

    train_scaled = (
        (train_matrix - mean) / std
    ).astype(np.float32)

    test_scaled = (
        (test_matrix - mean) / std
    ).astype(np.float32)

    # --------------------------------------------------------
    # Pretraining sequences
    # --------------------------------------------------------
    sequences = []

    for indices in split_segments_generic(
        train_static[TIME_COL],
        1,
    ):
        segment = train_scaled[indices]

        if len(segment) < H7_SSL_SEQUENCE_LENGTH:
            continue

        last_start = (
            len(segment)
            - H7_SSL_SEQUENCE_LENGTH
        )

        for start in range(
            0,
            last_start + 1,
            H7_SSL_STRIDE,
        ):
            sequences.append(
                segment[
                    start :
                    start + H7_SSL_SEQUENCE_LENGTH
                ].copy()
            )

    if not sequences:
        print(
            "H7 skipped: no contiguous sequence reaches "
            f"{H7_SSL_SEQUENCE_LENGTH} timesteps."
        )
        return None

    rng = np.random.RandomState(
        RANDOM_STATE
    )

    if len(sequences) > H7_SSL_MAX_SEQUENCES:
        selected = rng.choice(
            len(sequences),
            size=H7_SSL_MAX_SEQUENCES,
            replace=False,
        )
        sequences = [
            sequences[index]
            for index in selected
        ]

    sequence_array = np.asarray(
        sequences,
        dtype=np.float32,
    )

    if not np.isfinite(sequence_array).all():
        raise ValueError(
            "H7: non-finite values remain in SSL sequences."
        )

    sequence_tensor = torch.tensor(
        sequence_array,
        dtype=torch.float32,
    )

    data_generator = torch.Generator()
    data_generator.manual_seed(
        RANDOM_STATE
    )

    loader = DataLoader(
        TensorDataset(sequence_tensor),
        batch_size=H7_SSL_BATCH_SIZE,
        shuffle=True,
        generator=data_generator,
        drop_last=False,
    )

    # --------------------------------------------------------
    # Model
    # --------------------------------------------------------
    class MaskedGRU(nn.Module):
        def __init__(
            self,
            input_dim,
            hidden_dim,
        ):
            super().__init__()

            self.encoder = nn.GRU(
                input_size=input_dim,
                hidden_size=hidden_dim,
                num_layers=1,
                batch_first=True,
            )

            self.decoder = nn.Linear(
                hidden_dim,
                input_dim,
            )

        def forward(self, x):
            encoded, _ = self.encoder(x)
            reconstructed = self.decoder(
                encoded
            )
            return reconstructed, encoded

    device = torch.device(
        "cuda"
        if torch.cuda.is_available()
        else "cpu"
    )

    model = MaskedGRU(
        input_dim=train_scaled.shape[1],
        hidden_dim=H7_SSL_HIDDEN_DIM,
    ).to(device)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=H7_SSL_LEARNING_RATE,
    )

    history = []

    # --------------------------------------------------------
    # Masked reconstruction pretraining
    # --------------------------------------------------------
    for epoch in range(H7_SSL_EPOCHS):
        model.train()
        losses = []
        masked_losses = []
        full_losses = []

        for (batch,) in loader:
            batch = batch.to(device)

            mask = (
                torch.rand(
                    batch.shape,
                    device=device,
                )
                < H7_SSL_MASK_RATE
            )

            # Guarantee at least one masked value.
            if not bool(mask.any().item()):
                mask[
                    0,
                    -1,
                    0,
                ] = True

            masked_input = batch.clone()
            masked_input[mask] = 0.0

            reconstructed, _ = model(
                masked_input
            )

            masked_loss = (
                (
                    reconstructed[mask]
                    - batch[mask]
                )
                ** 2
            ).mean()

            full_loss = (
                (
                    reconstructed
                    - batch
                )
                ** 2
            ).mean()

            loss = (
                masked_loss
                + 0.10 * full_loss
            )

            if not bool(
                torch.isfinite(loss).item()
            ):
                raise FloatingPointError(
                    "H7: non-finite SSL loss."
                )

            optimizer.zero_grad(
                set_to_none=True
            )
            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=5.0,
            )

            optimizer.step()

            losses.append(
                float(
                    loss.detach()
                    .cpu()
                    .item()
                )
            )
            masked_losses.append(
                float(
                    masked_loss.detach()
                    .cpu()
                    .item()
                )
            )
            full_losses.append(
                float(
                    full_loss.detach()
                    .cpu()
                    .item()
                )
            )

        history.append(
            {
                "epoch": epoch + 1,
                "loss": float(
                    np.mean(losses)
                ),
                "masked_loss": float(
                    np.mean(masked_losses)
                ),
                "full_loss": float(
                    np.mean(full_losses)
                ),
                "n_sequences": int(
                    len(sequence_array)
                ),
                "device": str(device),
            }
        )

    # --------------------------------------------------------
    # Fixed-length causal embedding extraction
    # --------------------------------------------------------
    def build_context_windows(
        matrix,
        timestamps,
    ):
        """
        Creates one left-padded causal context per timestep.
        The final position of each context corresponds to the
        current timestep.
        """
        context_length = int(
            H7_SSL_SEQUENCE_LENGTH
        )
        windows = np.zeros(
            (
                len(matrix),
                context_length,
                matrix.shape[1],
            ),
            dtype=np.float32,
        )

        for indices in split_segments_generic(
            timestamps,
            1,
        ):
            segment = matrix[indices]

            for local_idx, global_idx in enumerate(
                indices
            ):
                start = max(
                    0,
                    local_idx
                    - context_length
                    + 1,
                )

                context = segment[
                    start :
                    local_idx + 1
                ]

                # Left pad using the first available timestep.
                pad_count = (
                    context_length
                    - len(context)
                )

                if pad_count > 0:
                    padding = np.repeat(
                        context[0:1],
                        pad_count,
                        axis=0,
                    )
                    context = np.vstack(
                        [padding, context]
                    )

                windows[global_idx] = context

        return windows

    def extract_embeddings(
        matrix,
        timestamps,
    ):
        contexts = build_context_windows(
            matrix,
            timestamps,
        )

        embedding_batches = []
        model.eval()

        with torch.no_grad():
            for start in range(
                0,
                len(contexts),
                H7_SSL_BATCH_SIZE,
            ):
                batch_context = torch.tensor(
                    contexts[
                        start :
                        start + H7_SSL_BATCH_SIZE
                    ],
                    dtype=torch.float32,
                    device=device,
                )

                _, encoded = model(
                    batch_context
                )

                # Current timestep is at the final sequence position.
                current_embedding = encoded[
                    :,
                    -1,
                    :
                ]

                embedding_batches.append(
                    current_embedding
                    .detach()
                    .cpu()
                    .numpy()
                )

        embeddings = np.vstack(
            embedding_batches
        ).astype(np.float32)

        if embeddings.shape != (
            len(matrix),
            H7_SSL_HIDDEN_DIM,
        ):
            raise ValueError(
                "H7: invalid embedding shape: "
                f"{embeddings.shape}"
            )

        return embeddings

    train_embedding = extract_embeddings(
        train_scaled,
        train_static[TIME_COL],
    )

    test_embedding = extract_embeddings(
        test_scaled,
        test_static[TIME_COL],
    )

    train_augmented = train_static.copy(
        deep=True
    )
    test_augmented = test_static.copy(
        deep=True
    )

    embedding_cols = []

    for dim in range(
        H7_SSL_HIDDEN_DIM
    ):
        column = (
            f"ssl_embedding_{dim:02d}"
        )
        embedding_cols.append(column)

        train_augmented[column] = (
            train_embedding[:, dim]
        )
        test_augmented[column] = (
            test_embedding[:, dim]
        )

    model_features = list(
        dict.fromkeys(
            list(bundle.selected_features)
            + embedding_cols
        )
    )

    missing_train_features = [
        column
        for column in model_features
        if column not in train_augmented.columns
    ]
    missing_test_features = [
        column
        for column in model_features
        if column not in test_augmented.columns
    ]

    if (
        missing_train_features
        or missing_test_features
    ):
        raise ValueError(
            "H7: missing model features. "
            f"train={missing_train_features[:5]}, "
            f"test={missing_test_features[:5]}"
        )

    probability = fit_xgb_ensemble(
        transform_features(
            train_augmented,
            model_features,
        ),
        train_augmented[
            TARGET_COL
        ].astype(str),
        transform_features(
            test_augmented,
            model_features,
        ),
        target_classes=bundle.model_classes,
        seeds=SEEDS,
        save_dir=None,
    )

    probability = smooth_probabilities_generic(
        probability,
        test_static[TIME_COL],
        1,
    )

    path = viterbi_generic(
        probability,
        test_static[TIME_COL],
        bundle.transition,
        1,
    )

    classes = np.asarray(
        bundle.model_classes,
        dtype=object,
    )

    predictions = classes[path].astype(str)

    if len(predictions) != len(test_static):
        raise ValueError(
            "H7: prediction length mismatch: "
            f"{len(predictions)} != {len(test_static)}"
        )

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {
        "predictions": predictions,
        "probabilities": probability,
        "training_history": pd.DataFrame(
            history
        ),
        "model_features": model_features,
        "ssl_input_features": features,
        "train_embedding_shape": tuple(
            train_embedding.shape
        ),
        "test_embedding_shape": tuple(
            test_embedding.shape
        ),
        "n_ssl_sequences": int(
            len(sequence_array)
        ),
        "device": str(device),
    }


In [ ]:
# ============================================================
# 9. EVALUATION HELPERS
# ============================================================

EXPERIMENT_DESCRIPTIONS = {
    "A3": "A3 selected static + smoothing + Viterbi",
    "H1": "Episode graph + graph jump repair",
    "H2": "Multi-window stable anchors",
    "H3": "Change-point anchor filling",
    "H4": "Explicit-duration pruned Graph-HSMM",
    "H5": "Room-conditional target-day calibration",
    "H6": "Segment boundary classifier",
    "H7": "Self-supervised causal GRU",
}


def evaluate_protocols_h(y_true, y_pred, train_classes, experiment_id, test_day):
    y_true = pd.Series(y_true).astype(str).to_numpy()
    y_pred = pd.Series(y_pred).astype(str).to_numpy()
    train_set = set(map(str, train_classes))
    masks = {
        "open_set": np.ones(len(y_true), dtype=bool),
        "closed_set": np.array([label in train_set for label in y_true]),
    }
    rows = []
    for protocol, mask in masks.items():
        true = y_true[mask]
        pred = y_pred[mask]
        labels = sorted(pd.unique(true))
        rows.append({
            "experiment_id": experiment_id,
            "description": EXPERIMENT_DESCRIPTIONS[experiment_id],
            "test_day": str(test_day),
            "protocol": protocol,
            "n_samples": int(mask.sum()),
            "macro_f1": f1_score(true, pred, labels=labels, average="macro", zero_division=0),
            "accuracy": accuracy_score(true, pred),
            "balanced_accuracy": balanced_accuracy_score(true, pred),
            "weighted_f1": f1_score(true, pred, labels=labels, average="weighted", zero_division=0),
            "unseen_support": int(sum(label not in train_set for label in true)),
        })
    return pd.DataFrame(rows)


def evaluate_class_h(y_true, y_pred, train_classes, experiment_id, test_day):
    y_true = pd.Series(y_true).astype(str).to_numpy()
    y_pred = pd.Series(y_pred).astype(str).to_numpy()
    train_set = set(map(str, train_classes))
    mask = np.array([label in train_set for label in y_true])
    true = y_true[mask]
    pred = y_pred[mask]
    labels = sorted(pd.unique(true))
    precision, recall, f1, support = precision_recall_fscore_support(true, pred, labels=labels, zero_division=0)
    return pd.DataFrame({
        "experiment_id": experiment_id,
        "test_day": str(test_day),
        "class": labels,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "support": support,
    })


def hallway_intervals(labels, timestamps):
    labels = pd.Series(labels).astype(str).to_numpy()
    intervals = []
    for indices in split_segments_generic(timestamps, 1):
        start = None
        for i in range(len(indices) + 1):
            active = i < len(indices) and labels[indices[i]] == HALLWAY_LABEL
            if active and start is None:
                start = i
            if not active and start is not None:
                intervals.append((int(indices[start]), int(indices[i - 1])))
                start = None
    return intervals


def interval_iou(a, b):
    intersection = max(0, min(a[1], b[1]) - max(a[0], b[0]) + 1)
    union = (a[1] - a[0] + 1) + (b[1] - b[0] + 1) - intersection
    return intersection / max(union, 1)


def evaluate_episode_h(y_true, y_pred, timestamps, experiment_id, test_day, iou_threshold=0.10):
    true_intervals = hallway_intervals(y_true, timestamps)
    pred_intervals = hallway_intervals(y_pred, timestamps)
    candidates = []
    for ti, true_interval in enumerate(true_intervals):
        for pi, pred_interval in enumerate(pred_intervals):
            iou = interval_iou(true_interval, pred_interval)
            if iou >= iou_threshold:
                candidates.append((iou, ti, pi))
    candidates.sort(reverse=True)
    matched_true, matched_pred, matched = set(), set(), []
    for iou, ti, pi in candidates:
        if ti in matched_true or pi in matched_pred:
            continue
        matched_true.add(ti)
        matched_pred.add(pi)
        matched.append((iou, true_intervals[ti], pred_intervals[pi]))
    tp = len(matched)
    fp = len(pred_intervals) - tp
    fn = len(true_intervals) - tp
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    episode_f1 = 2 * precision * recall / max(precision + recall, 1e-12)
    return pd.DataFrame([{
        "experiment_id": experiment_id,
        "test_day": str(test_day),
        "true_episodes": len(true_intervals),
        "predicted_episodes": len(pred_intervals),
        "matched_episodes": tp,
        "episode_precision": precision,
        "episode_recall": recall,
        "episode_f1": episode_f1,
        "mean_matched_iou": np.mean([x[0] for x in matched]) if matched else np.nan,
        "mean_onset_error_seconds": np.mean([abs(x[2][0] - x[1][0]) for x in matched]) if matched else np.nan,
        "mean_offset_error_seconds": np.mean([abs(x[2][1] - x[1][1]) for x in matched]) if matched else np.nan,
    }])


In [ ]:
# ============================================================
# 10. RUN ONE OUTER FOLD
# ============================================================

def run_hallway_fold(test_day):
    fold_dir = OUTPUT_DIR / f"fold_{test_day}"
    fold_dir.mkdir(parents=True, exist_ok=True)

    train_1s = df[df[DATE_COL].astype(str) != str(test_day)].copy().sort_values([DATE_COL, TIME_COL]).reset_index(drop=True)
    test_1s = df[df[DATE_COL].astype(str) == str(test_day)].copy().sort_values([TIME_COL]).reset_index(drop=True)
    train_classes = sorted(train_1s[TARGET_COL].astype(str).unique())

    bundle, train_static = fit_a3_bundle_generic(train_1s, WINDOW_SECONDS, SEEDS, RANDOM_STATE)
    a3 = predict_a3_bundle_generic(bundle, test_1s)
    test_static = a3["static"]
    predictions = {"A3": np.asarray(a3["predictions"], dtype=object)}

    graph, train_episodes = build_hallway_graph(train_1s)
    test_episodes = extract_label_episodes(test_1s)
    train_episodes.to_csv(fold_dir / "train_episodes.csv", index=False)
    test_episodes.to_csv(fold_dir / "test_episodes.csv", index=False)

    graph_rows = []
    for room, neighbors in graph.adjacency.items():
        for neighbor in neighbors:
            if str(room) < str(neighbor):
                graph_rows.append({
                    "room_a": room,
                    "room_b": neighbor,
                    "route_count": graph.route_counts.get((room, neighbor), 0),
                    "direct_count": graph.direct_counts.get((room, neighbor), 0),
                })
    pd.DataFrame(graph_rows).to_csv(fold_dir / "train_graph_edges.csv", index=False)

    if "H1" in RUN_EXPERIMENTS:
        predictions["H1"] = h1_graph_jump_repair(
            predictions["A3"],
            a3["top1_confidence"],
            test_1s[TIME_COL],
            graph,
        )

    aligned_results = {}
    for seconds in H2_LONG_WINDOWS:
        long_data = MULTI_WINDOW_DATA[seconds]
        train_long = long_data[long_data[DATE_COL].astype(str) != str(test_day)].copy().reset_index(drop=True)
        test_long = long_data[long_data[DATE_COL].astype(str) == str(test_day)].copy().reset_index(drop=True)
        long_bundle, _ = fit_a3_bundle_generic(train_long, seconds, LONG_WINDOW_SEEDS, RANDOM_STATE + seconds)
        long_result = predict_a3_bundle_generic(long_bundle, test_long)
        aligned_results[seconds] = align_long_result(test_1s, test_long, long_result, seconds)

    anchors = build_multi_window_anchors(test_1s, aligned_results)
    anchors.to_csv(fold_dir / "multi_window_anchors.csv", index=False)
    predictions["H2"] = h2_anchor_stabilization(predictions["A3"], anchors)

    train_transition = add_causal_transition_features(train_static)
    test_transition = add_causal_transition_features(test_static)
    change_score = compute_change_score(
        test_transition,
        fit_change_scaler(train_transition),
        a3["entropy"],
        a3["probability_margin"],
    )

    if "H3" in RUN_EXPERIMENTS:
        predictions["H3"], audit = h3_change_point_fill(
            predictions["H2"], anchors, test_1s[TIME_COL], change_score, graph
        )
        audit.to_csv(fold_dir / "H3_gap_audit.csv", index=False)

    if "H4" in RUN_EXPERIMENTS:
        predictions["H4"], audit = h4_graph_hsmm(
            predictions["H2"], anchors, test_1s[TIME_COL], change_score,
            a3["probabilities_smooth"], a3["classes"], graph
        )
        audit.to_csv(fold_dir / "H4_gap_audit.csv", index=False)

    if "H5" in RUN_EXPERIMENTS and H5_ENABLE_TRANSDUCTIVE_ADAPTATION:
        offsets, audit = estimate_room_offsets(train_1s, test_1s, a3)
        audit.to_csv(fold_dir / "H5_offsets.csv", index=False)
        calibrated = apply_room_offsets(test_1s, a3["predictions"], offsets)
        predictions["H5"] = np.asarray(
            predict_a3_bundle_generic(bundle, calibrated)["predictions"],
            dtype=object,
        )

    if "H6" in RUN_EXPERIMENTS:
        boundary_features = get_boundary_features(train_transition)
        threshold, threshold_audit = select_boundary_threshold_lodo(train_transition, boundary_features)
        threshold_audit.to_csv(fold_dir / "H6_threshold_search.csv", index=False)
        boundary_model, weight = fit_boundary_model(train_transition, boundary_features)
        boundary_probability = boundary_model.predict_proba(transform_features(test_transition, boundary_features))[:, 1]
        boundary_probability = smooth_binary_probability(boundary_probability, test_1s[TIME_COL], 3)
        predictions["H6"], audit = h6_boundary_decode(
            predictions["H2"], anchors, test_1s[TIME_COL], boundary_probability, threshold, graph
        )
        audit["selected_threshold"] = threshold
        audit["positive_weight"] = weight
        audit.to_csv(fold_dir / "H6_gap_audit.csv", index=False)

    if "H7" in RUN_EXPERIMENTS:
        ssl = run_ssl_temporal_experiment(bundle, train_static, test_static)
        if ssl is not None:
            predictions["H7"] = np.asarray(ssl["predictions"], dtype=object)
            ssl["training_history"].to_csv(
                fold_dir / "H7_training_history.csv",
                index=False,
            )

            pd.DataFrame(
                {
                    "feature": ssl["model_features"],
                }
            ).to_csv(
                fold_dir / "H7_model_features.csv",
                index=False,
            )

            pd.DataFrame(
                {
                    "feature": ssl["ssl_input_features"],
                }
            ).to_csv(
                fold_dir / "H7_ssl_input_features.csv",
                index=False,
            )

            (
                fold_dir / "H7_diagnostics.json"
            ).write_text(
                json.dumps(
                    {
                        "n_ssl_sequences": int(
                            ssl["n_ssl_sequences"]
                        ),
                        "device": str(
                            ssl["device"]
                        ),
                        "train_embedding_shape": list(
                            ssl["train_embedding_shape"]
                        ),
                        "test_embedding_shape": list(
                            ssl["test_embedding_shape"]
                        ),
                    },
                    indent=2,
                    ensure_ascii=False,
                ),
                encoding="utf-8",
            )

    prediction_df = test_1s[[TIME_COL, DATE_COL, TARGET_COL]].copy().rename(columns={TARGET_COL: "y_true"})
    metrics, classes, episodes = [], [], []
    for experiment_id, y_pred in predictions.items():
        prediction_df[f"y_pred_{experiment_id}"] = np.asarray(y_pred, str)
        metrics.append(evaluate_protocols_h(prediction_df["y_true"], y_pred, train_classes, experiment_id, test_day))
        classes.append(evaluate_class_h(prediction_df["y_true"], y_pred, train_classes, experiment_id, test_day))
        episodes.append(evaluate_episode_h(prediction_df["y_true"], y_pred, test_1s[TIME_COL], experiment_id, test_day))

    metrics = pd.concat(metrics, ignore_index=True)
    classes = pd.concat(classes, ignore_index=True)
    episodes = pd.concat(episodes, ignore_index=True)

    prediction_df.to_csv(fold_dir / "predictions.csv", index=False)
    metrics.to_csv(fold_dir / "metrics.csv", index=False)
    classes.to_csv(fold_dir / "class_metrics.csv", index=False)
    episodes.to_csv(fold_dir / "episode_metrics.csv", index=False)
    return metrics, classes, episodes


In [ ]:
# ============================================================
# 11. RUN FOUR LODO FOLDS
# ============================================================

metric_parts, class_parts, episode_parts = [], [], []
for test_day in sorted(df[DATE_COL].astype(str).unique()):
    print("=" * 100)
    print("OUTER TEST DAY:", test_day)
    print("=" * 100)
    fold_dir = OUTPUT_DIR / f"fold_{test_day}"

    if RESUME_COMPLETED_FOLDS and (fold_dir / "metrics.csv").exists():
        metrics = pd.read_csv(fold_dir / "metrics.csv")
        classes = pd.read_csv(fold_dir / "class_metrics.csv")
        episodes = pd.read_csv(fold_dir / "episode_metrics.csv")
    else:
        metrics, classes, episodes = run_hallway_fold(str(test_day))

    metric_parts.append(metrics)
    class_parts.append(classes)
    episode_parts.append(episodes)
    display(
        metrics[metrics["protocol"] == "closed_set"]
        [["experiment_id", "macro_f1", "accuracy", "balanced_accuracy", "weighted_f1"]]
        .sort_values("macro_f1", ascending=False)
    )

all_metrics_df = pd.concat(metric_parts, ignore_index=True)
all_class_metrics_df = pd.concat(class_parts, ignore_index=True)
all_episode_metrics_df = pd.concat(episode_parts, ignore_index=True)

all_metrics_df.to_csv(OUTPUT_DIR / "all_fold_metrics.csv", index=False)
all_class_metrics_df.to_csv(OUTPUT_DIR / "all_class_metrics.csv", index=False)
all_episode_metrics_df.to_csv(OUTPUT_DIR / "all_episode_metrics.csv", index=False)


In [ ]:
# ============================================================
# 12. FINAL SUMMARY
# ============================================================

summary = (
    all_metrics_df.groupby(["experiment_id", "description", "protocol"])
    .agg(
        mean_macro_f1=("macro_f1", "mean"),
        std_macro_f1=("macro_f1", "std"),
        mean_accuracy=("accuracy", "mean"),
        mean_balanced_accuracy=("balanced_accuracy", "mean"),
        mean_weighted_f1=("weighted_f1", "mean"),
        total_unseen_support=("unseen_support", "sum"),
        folds=("test_day", "nunique"),
    )
    .reset_index()
)

key_class_summary = (
    all_class_metrics_df[all_class_metrics_df["class"].isin(KEY_CLASSES)]
    .groupby(["experiment_id", "class"])
    .agg(
        mean_precision=("precision", "mean"),
        mean_recall=("recall", "mean"),
        mean_f1=("f1", "mean"),
        total_support=("support", "sum"),
    )
    .reset_index()
)

episode_summary = (
    all_episode_metrics_df.groupby("experiment_id")
    .agg(
        mean_episode_precision=("episode_precision", "mean"),
        mean_episode_recall=("episode_recall", "mean"),
        mean_episode_f1=("episode_f1", "mean"),
        mean_matched_iou=("mean_matched_iou", "mean"),
        mean_onset_error_seconds=("mean_onset_error_seconds", "mean"),
        mean_offset_error_seconds=("mean_offset_error_seconds", "mean"),
        total_true_episodes=("true_episodes", "sum"),
        total_predicted_episodes=("predicted_episodes", "sum"),
    )
    .reset_index()
)

closed_summary = summary[summary["protocol"] == "closed_set"].sort_values("mean_macro_f1", ascending=False)
hallway_summary = key_class_summary[key_class_summary["class"] == HALLWAY_LABEL].sort_values("mean_f1", ascending=False)

summary.to_csv(OUTPUT_DIR / "experiment_summary.csv", index=False)
closed_summary.to_csv(OUTPUT_DIR / "closed_set_summary.csv", index=False)
key_class_summary.to_csv(OUTPUT_DIR / "key_class_summary.csv", index=False)
hallway_summary.to_csv(OUTPUT_DIR / "hallway_summary.csv", index=False)
episode_summary.to_csv(OUTPUT_DIR / "episode_summary.csv", index=False)

print("Closed-set summary")
display(closed_summary)
print("Hallway class summary")
display(hallway_summary)
print("Episode summary")
display(episode_summary.sort_values("mean_episode_f1", ascending=False))

plt.figure(figsize=(11, 6))
plt.bar(closed_summary["experiment_id"], closed_summary["mean_macro_f1"])
plt.errorbar(
    closed_summary["experiment_id"],
    closed_summary["mean_macro_f1"],
    yerr=closed_summary["std_macro_f1"].fillna(0.0),
    fmt="none",
    capsize=4,
)
plt.xlabel("Experiment")
plt.ylabel("Mean closed-set Macro-F1")
plt.title("A3 and seven hallway experiments")
plt.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "closed_macro_f1_comparison.png", dpi=200, bbox_inches="tight")
plt.show()
plt.close()

plt.figure(figsize=(11, 6))
plt.bar(hallway_summary["experiment_id"], hallway_summary["mean_f1"])
plt.xlabel("Experiment")
plt.ylabel("Mean hallway F1")
plt.title("Hallway window-level F1")
plt.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "hallway_window_f1_comparison.png", dpi=200, bbox_inches="tight")
plt.show()
plt.close()

plt.figure(figsize=(11, 6))
plot_episode = episode_summary.sort_values("mean_episode_f1", ascending=False)
plt.bar(plot_episode["experiment_id"], plot_episode["mean_episode_f1"])
plt.xlabel("Experiment")
plt.ylabel("Mean hallway episode F1")
plt.title("Hallway episode-level detection")
plt.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "hallway_episode_f1_comparison.png", dpi=200, bbox_inches="tight")
plt.show()
plt.close()


## Interpretation rule

MÃƒÂ¡Ã‚Â»Ã¢â€žÂ¢t experiment chÃƒÂ¡Ã‚Â»Ã¢â‚¬Â° Ãƒâ€žÃ¢â‚¬ËœÃƒâ€ Ã‚Â°ÃƒÂ¡Ã‚Â»Ã‚Â£c xem lÃƒÆ’Ã‚Â  cÃƒÂ¡Ã‚ÂºÃ‚Â£i thiÃƒÂ¡Ã‚Â»Ã¢â‚¬Â¡n hallway khi Ãƒâ€žÃ¢â‚¬ËœÃƒÂ¡Ã‚Â»Ã¢â‚¬Å“ng thÃƒÂ¡Ã‚Â»Ã‚Âi:

1. Hallway window-level F1 tÃƒâ€žÃ†â€™ng.
2. Episode F1 hoÃƒÂ¡Ã‚ÂºÃ‚Â·c segment IoU tÃƒâ€žÃ†â€™ng.
3. Closed-set macro-F1 tÃƒÂ¡Ã‚Â»Ã¢â‚¬Â¢ng khÃƒÆ’Ã‚Â´ng giÃƒÂ¡Ã‚ÂºÃ‚Â£m Ãƒâ€žÃ¢â‚¬ËœÃƒÆ’Ã‚Â¡ng kÃƒÂ¡Ã‚Â»Ã†â€™.
4. KÃƒÂ¡Ã‚ÂºÃ‚Â¿t quÃƒÂ¡Ã‚ÂºÃ‚Â£ khÃƒÆ’Ã‚Â´ng chÃƒÂ¡Ã‚Â»Ã¢â‚¬Â° Ãƒâ€žÃ¢â‚¬ËœÃƒÂ¡Ã‚ÂºÃ‚Â¿n tÃƒÂ¡Ã‚Â»Ã‚Â« mÃƒÂ¡Ã‚Â»Ã¢â€žÂ¢t outer fold.
5. SÃƒÂ¡Ã‚Â»Ã¢â‚¬Ëœ predicted hallway episodes khÃƒÆ’Ã‚Â´ng tÃƒâ€žÃ†â€™ng bÃƒÂ¡Ã‚ÂºÃ‚Â¥t thÃƒâ€ Ã‚Â°ÃƒÂ¡Ã‚Â»Ã‚Âng.

H5 phÃƒÂ¡Ã‚ÂºÃ‚Â£i Ãƒâ€žÃ¢â‚¬ËœÃƒâ€ Ã‚Â°ÃƒÂ¡Ã‚Â»Ã‚Â£c bÃƒÆ’Ã‚Â¡o cÃƒÆ’Ã‚Â¡o riÃƒÆ’Ã‚Âªng lÃƒÆ’Ã‚Â  **transductive unsupervised adaptation**.
